# Track S / Batch 3b cache: sl16 (16 slices per slot, 130 mm, legacy rules)

E13-train cells 0-17 + 32 (helpers only, ARM_MODE=s0 no-op), then one CPU cache-build cell writing the cache in train.csv order through a memmap (14 GB).


# V41: DINOv3 + E10 alpha 0.60

This version changes only the precommitted E10 configuration from uniform 0.35 to uniform 0.60. Grouped nested selection chose 0.60 in all five public outer folds; an independent OOF source chose 0.70 in all five. No visible-test tuning or identifier is used.


In [ ]:
import pandas as _pd, numpy as _np, os as _os
_ORIG = _pd.read_csv
_T = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
_SRC = {'report_labels_v2.csv','llm_labels_v2.csv','labels_llm_gpt56sol.csv'}
_CACHE = {}
def _find(name):
    for root, dirs, files in _os.walk('/kaggle/input'):
        dirs[:] = [d for d in dirs if d not in ('train_series','test_series')]
        if name in files:
            return _os.path.join(root, name)
    raise FileNotFoundError(name)
def _mean5():
    if 'y' in _CACHE:
        return _CACHE['y']
    uid = 'StudyInstanceUID'
    frames = []
    for n in ['report_labels_v2.csv','llm_labels_v2.csv','labels_llm_gpt56sol.csv','llm_labels_v4_blend.csv']:
        f = _ORIG(_find(n))
        frames.append(f[[uid]+_T].drop_duplicates(uid).set_index(uid))
    q = _ORIG(_find('weak_labels_v2_filled.csv'))
    q = q[[uid]+[t+'__confidence' for t in _T]].rename(columns={t+'__confidence': t for t in _T}).drop_duplicates(uid).set_index(uid)
    frames.append(q)
    idx = frames[1].index
    cube = _np.stack([fr.reindex(idx)[_T].to_numpy(float) for fr in frames], 0)
    y = _np.nanmean(cube, 0)
    out = _pd.DataFrame(y, columns=_T, index=idx).reset_index()
    print('[5-teacher patch] cube', cube.shape, '| nan per teacher', [int(_np.isnan(c).sum()) for c in cube], '| y mean', round(float(_np.nanmean(y)), 4), flush=True)
    _CACHE['y'] = out
    return out
def _patched(path, *a, **k):
    if isinstance(path, (str, _os.PathLike)) and _os.path.basename(str(path)) in _SRC:
        print('[5-teacher patch] intercept', _os.path.basename(str(path)), flush=True)
        return _mean5().copy()
    return _ORIG(path, *a, **k)
_pd.read_csv = _patched
print('[5-teacher patch] installed: 3 report sources -> mean of 5 teachers (pilkwang, hans_v2, sol56, hans_v4_blend, qwen confidence)')


# V40: DINOv3 cross-series + E10 RadImageNet hybrid

This hidden-test-safe candidate retains Mattia Angeli's cross-series DINOv3 member, removes its older Rad15 stage, and applies fishface's hash-pinned E10 uniform-0.35 RadImageNet correction to the resulting dynamic DINOv2+DINOv3 parent. No visible test identifier or prediction is embedded, and no test-time weight selection is performed.


# Bend the Knee Ensemble

A community pipeline, with one more model added.

## Thank you

This is built on work other people did first and shared:

- **pilkwang** — twenty trained models that are part of this ensemble, and a
  set of labels read from the reports.
- **stevenleehans** and **lixin73** — two more sets of labels read from the
  reports, so one reading could be checked against another.
- **tonylica** — four more trained models that join the ensemble.
- **marwanmath** — the official RadImageNet ResNet-50 weights.
- **prvsiyan** — the notebook this was forked from (Apache 2.0). Most of its
  later stages have been removed here.
- **cf696666** — for leaving two findings out of the RadImageNet blend, which
  is done here too.

### The added member

The submission this forks from is a rank mean of twenty **DINOv2** models. Added
here is one **DINOv3 ViT-S/16** — self-supervised and pretrained without labels,
then fine-tuned on this competition's knee MRI — rank-blended into that
family.

Above the encoder, each series carries a learned type embedding — plane crossed
with fat suppression — added to all of its tokens. The tokens of every series in
a study are then concatenated into one key/value sequence, and twelve learned
queries, one per finding, cross-attend over it with multi-head attention. A
finding therefore draws evidence from any series in the study at once, rather
than from per-series summaries combined afterwards. Each query's output is
concatenated with the mean and the max of the per-series CLS embeddings before
the classifier. Series a study does not contain are removed by the key-padding
mask, so nothing is imputed for them.

**RadImageNet** joins the vote at the end — a ResNet-50 pretrained on
radiology images rather than natural ones. Its heads were retrained here
rather than used as published.

Its five folds are combined on ranks rather than by averaging probabilities, matching
how the other two families are already combined — the metric is macro ROC-AUC, so only
ordering matters. That correction is due to **romantamrazov** (*RSNA Knee | DINOsaur V2*).

In [ ]:
from __future__ import annotations
import re
import unicodedata
TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
_PRE = str.maketrans({'ı': 'i', 'İ': 'i', 'I': 'i', 'ß': 'ss', 'đ': 'd', 'Đ': 'd', 'ø': 'o', 'Ø': 'o', 'æ': 'ae', 'Æ': 'ae'})

def normalize(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.translate(_PRE).lower()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join((ch for ch in text if not unicodedata.combining(ch)))
    text = text.replace('\xad', '')
    text = re.sub('[_\\-/\\\\]+', ' ', text)
    text = re.sub('[ \\t]+', ' ', text)
    return text
_SENT_SPLIT = re.compile('(?<=[.;!?])\\s+|\\n+')

def unwrap(text: str) -> str:
    if not isinstance(text, str):
        return ''
    out = []
    for line in text.split('\n'):
        s = line.strip()
        if out and out[-1] and (not re.search('[.;:!?>*•]$', out[-1])) and (len(out[-1].split()) >= 4) and s and (not s[:1].isupper()):
            out[-1] = out[-1] + ' ' + s
        else:
            out.append(s)
    return '\n'.join(out)

def clauses(text: str):
    norm = normalize(unwrap(text) if FEATURES['unwrap'] else text)
    raw = [c.strip() for c in _SENT_SPLIT.split(norm) if c and c.strip()]
    merged = []
    for i, c in enumerate(raw):
        if c.endswith(':') and len(c.split()) <= 14 and (i + 1 < len(raw)):
            merged.append(c + ' ' + raw[i + 1])
        merged.append(c)
    out = []
    for c in merged:
        out.append(c)
        if len(c.split()) > 25:
            out.extend((p.strip() for p in c.split(',') if len(p.split()) > 2))
    return out
FEATURES = {'unwrap': True, 'directional_negation': True, 'oa_inherit': True, 'graded_pathology': True, 'synovitis_backoff': True}

def _rx(*alts: str) -> re.Pattern:
    return re.compile('|'.join(alts))
PRE_NEG = _rx('\\bno\\b', '\\bnot\\b', '\\bwithout\\b', '\\bnegative for\\b', '\\babsence\\b', '\\bno evidence\\b', '\\bfree of\\b', '\\bnone\\b', '\\bneither\\b', '\\bnor\\b', '\\bsin\\b', '\\bno hay\\b', '\\bausencia\\b', '\\bausentes?\\b', '\\bno se\\b', '\\bpas de\\b', '\\bsans\\b', '\\baucune?\\b', '\\bgeen\\b', '\\bzonder\\b', '\\bniet\\b', '\\bkeine?[nmrs]?\\b', '\\bohne\\b', '\\bnicht\\b', '\\bkein\\b', '\\bnema\\b', '\\bbez\\b', '\\bnisu\\b', '\\bnije\\b', '\\bδεν\\b', '\\bχωρις\\b', 'ουδεν', '\\bουτε\\b', '\\bбез\\b', '\\bне\\b', 'липсва', '\\bняма\\b')
POST_NEG = _rx('\\byok\\b', '\\byoktur\\b', 'izlenmemekte', 'saptanmadi', '\\bdegil\\b', 'gozlenmemekte', 'mevcut degil', 'eslik etmiyor', '\\bizlenmedi\\b', 'izlenmemistir', 'saptanmamistir', 'gorulmemistir', '\\bnema znakova\\b', 'bez znakova')
NEGATION = _rx(PRE_NEG.pattern, POST_NEG.pattern, '\\bunremarkable\\b')
NEG_WINDOW = 90

def _negated(clause: str, start: int, end: int) -> bool:
    for m in PRE_NEG.finditer(clause):
        if m.end() <= start and start - m.end() <= NEG_WINDOW:
            if not re.search('\\b(but|however|ancak|fakat|pero|maar|aber|no i|ali|ωστοσο|αλλα|но)\\b', clause[m.end():start]):
                return True
    for m in POST_NEG.finditer(clause):
        if m.start() >= end and m.start() - end <= NEG_WINDOW:
            return True
    return False
NORMALITY = _rx('\\bnormal', '\\bintact\\b', '\\bpreserved\\b', '\\bwithin normal limits\\b', 'limites normales', '\\bconservad', '\\bintegr', '\\bnormales\\b', '\\bdoga(l|ll)\\b', 'korunmus', '\\bnormaldir\\b', 'olagan', '\\buredn', '\\bocuvan', '\\bodrzan', '\\bintakt', '\\bprimjeren', '\\bodrzanog kontinuiteta', '\\bodržan', 'φυσιολογικ', 'ακεραι', 'δεν παρατηρουνται', 'δεν σημειωνονται', 'unauffallig', 'regelrecht', '\\bo\\.?b\\.?\\b', 'нормал', 'запазен', 'съхранен', '\\bбез особености\\b', 'интактн', '\\bgaaf\\b', '\\bnormaal\\b')
NORMAL_PHRASE = _rx('\\bsin alteracion', '\\bsin cambios\\b', '\\bsin particularidad', '\\bsin hallazgos\\b', '\\bsin lesion', '\\bsin signos de (rotura|lesion)', '\\bcontinu[oa]s?\\b', '\\bcontinuidad conservada\\b', '\\bno abnormalit', '\\bno significant abnormalit', '\\bunremarkable\\b', '\\bno evidence of (tear|injury|abnormalit)', '\\bohne auffalligkeit', '\\bkein nachweis\\b', '\\bohne befund\\b', '\\bgeen afwijking', '\\bzonder afwijking', '\\bsans anomalie', "\\bpas d[e']anomalie", '\\bbez osobitosti\\b', '\\bbez znakova (rupture|lezije)\\b', '\\bbez patoloskih\\b', 'χωρις αλλοιωσ', 'χωρις παθολογ', 'δεν παρατηρουνται (αξιολογα|παθολογ)', '\\bбез особености\\b', '\\bбез патологич', '\\bбез данни за\\b', '\\bozel bir ozellik yok', '\\bpatolojik bulgu (yok|izlenmemis)')
UNCERTAIN = _rx('\\bpossible\\b', '\\bprobable\\b', '\\bsuspicious\\b', '\\bsuspected?\\b', 'cannot (be )?exclude', '\\bmay\\b', '\\bquestionable\\b', '\\bequivocal\\b', '\\br/o\\b', '\\bdd\\b', '\\blikely\\b', '\\bsuggest', '\\bcompatible with\\b', '\\bposible\\b', 'sin criterios categoricos', '\\bdudos', '\\bsugier', '\\bmuhtemel\\b', '\\bolasi\\b', '\\bsupheli\\b', '\\bizlenim', '\\bdusundur', '\\bmoguce\\b', '\\bvjerojatno\\b', '\\bsumnja\\b', '\\bmoze odgovarati\\b', 'πιθαν', 'υποπτ', '\\bmoglich', '\\bverdachtig', '\\bfraglich', '\\bv\\.?a\\.?\\b', '\\bwohl\\b', '\\bвъзможно\\b', '\\bвероятно\\b', 'суспект', '\\bmogelijk\\b', '\\bverdacht\\b')

In [ ]:
TEAR = _rx('\\btear', '\\btorn\\b', '\\brupture', '\\bdisruption\\b', 'discontinuit', '\\bavuls', '\\bmacerat', '\\bbuckethandle\\b', 'bucket handle', '\\brotura\\b', '\\broturas\\b', '\\bruptura', '\\bdesgarro', '\\broto\\b', '\\bdechirure', '\\bdechire', '\\bscheur', '\\bruptuur', 'gescheurd', '\\briss\\b', 'einriss', '\\bruptur', 'zerreiss', '\\blasion', '\\bausriss', '\\byirtik', '\\byirtig', '\\bkopma\\b', 'butunluk kaybi', '\\brupturu\\b', 'devamsizlik', '\\brupture\\b', '\\bdevamliligi secilememis', '\\bpuknuce', '\\bprekid\\b', '\\bpukotin', '\\bruptur', 'ρηξη', 'ρηξις', 'ρηγμα', 'ασυνεχεια', 'руптура', 'разкъсв', 'разрив', 'скъсв', '\\bлезия\\b')
DEGEN = _rx('degenerat', '\\bmucoid\\b', '\\bmyxoid\\b', '\\bfray', '\\bfissur', 'dejeneratif', '\\bmukoid\\b', 'degenerativn', 'εκφυλ', 'дегенерат', '\\bμυξοειδ', '\\bμυξωδ', '\\bmeniskopat', '\\bmeniscopath', '\\bmuco ?ide\\b', 'aufgefasert', '\\bdejenerasyon\\b')
INJURY = _rx('\\binjur', '\\bsprain', '\\blesion', '\\blasion', '\\bedema\\b', '\\boedema\\b', '\\bodem\\b', '\\bedem\\b', '\\bοιδημα', '\\bодем', '\\bедем', '\\bstrain\\b', '\\bhigh signal\\b', '\\bsignal alteration\\b', '\\bhiperintens', '\\bhyperintens', 'aumento de senal', 'alteracion de senal', 'cambio de senal', '\\bsignalanhebung', '\\bsignalalteration', 'verhoogd signaal', 'sinyal artis', 'αυξημενο σημα', 'повишен сигнал', '\\besguince\\b', '\\bthicken', '\\bzadebljanje\\b', '\\bverdikking\\b', '\\bdistenzij', '\\blaksite\\b', '\\blaxity\\b', '\\bpartial\\b', '\\bparcijaln', '\\bparcial', '\\bpartiel', '\\bpartiell')
_GRADE_RX = re.compile('(?:grade|grad|grado|grau|derece|stupnja|stupanj|βαθμ|степен|icrs|outerbridge)[\\s:]*(?:grade\\s*)?([1-4]|iv|iii|ii|i)\\b')
_ROMAN = {'i': 1, 'ii': 2, 'iii': 3, 'iv': 4}

def _grade_of(clause: str):
    best = None
    for m in _GRADE_RX.finditer(clause):
        v = m.group(1)
        n = _ROMAN.get(v, None) if not v.isdigit() else int(v)
        if n is not None and (best is None or n > best):
            best = n
    return best
ANAT = {'ACL': _rx('anterior cruciate', '\\bacl\\b', 'cruzado anterior', '\\blca\\b', 'croise anterieur', 'voorste kruisband', '\\bvkb\\b', 'vorderes kreuzband', 'vorderen kreuzband', 'vordere kreuzband', 'on capraz', '\\bocb\\b', 'anterior capraz', 'prednji krizni', 'prednjeg krizn', 'προσθι[οα][^ ]* χιαστ', 'προσθιου χιαστου', 'χιαστο[^ ]* συνδεσμ', '\\bχιαστ\\w*', 'предна кръстна', 'предната кръстна', 'предна кръста', 'cruciate ligaments', 'ligamentos cruzados', 'ligaments croises', 'kruisbanden', 'kreuzbander', 'capraz baglar', 'krizn[a-z]* ligament[a-z]*', 'χιαστοι συνδεσμ', 'χιαστων συνδεσμ', 'кръстните връзки', 'кръстни връзки'), 'MCL': _rx('medial collateral', '\\bmcl\\b', 'tibial collateral', 'colateral medial', 'colateral interno', '\\blcm\\b', 'collateral medial', 'collateral interne', 'mediale collaterale', 'binnenband', '\\b(mediale|laterale) banden\\b', '\\bcollaterale banden\\b', 'innenband', 'mediales? kollateral', '\\bic yan bag', 'medial kollateral', '\\biyb\\b', 'medyal kollateral', 'medijalni kolateraln', 'medijalnog kolateraln', 'εσω πλαγι', 'εσωτερικο πλαγι', '\\bπλαγι\\w* συνδεσμ', '\\bπλαγιοι\\b', 'медиален колатерал', 'вътрешна странична', '\\bколатерал\\w*', '\\bcolaterales\\b', '\\bcollateraux\\b', '\\bcollateralen\\b', '\\bkolateralni\\b', 'collateral ligaments', 'ligamentos colaterales', 'ligaments collateraux', 'collaterale banden', 'kollateralbander', 'seitenbander', 'yan baglar', 'kolateraln[a-z]* ligament[a-z]*', 'πλαγιοι συνδεσμ', 'πλαγιων συνδεσμ', 'колатерални връзки', 'страничните връзки'), 'Medial Meniscus': _rx('medial meniscus', '\\bmm\\b(?= tear)', 'medial menisc', 'menisco medial', 'menisco interno', 'menisque medial', 'menisque interne', 'mediale meniscus', 'binnenmeniscus', 'innenmeniskus', 'medialen? meniskus', 'innenmeniskushinterhorn', 'medyal menisk', '\\bic menisk', 'medijalni meniskus', 'medijalnog meniskusa', 'medijalnom meniskusu', 'medijaln\\w* menisk\\w*', '\\bmedijalnog meniska\\b', 'medijalni menisk', 'εσω μηνισκ', 'μηνισκ[^ ]* του εσω', 'εσω διαμερισμα[^.]{0,40}μηνισκ', 'медиалния менискус', 'медиален менискус', 'вътрешния менискус', 'oba meniska', 'both menisci', 'ambos meniscos', 'beide menisci', 'her iki menisku', 'amfoteroi\\w* mhnisk', 'αμφοτερ\\w* μηνισκ', 'двата менискуса', 'medial (and|&) lateral menisc'), 'Lateral Meniscus': _rx('lateral meniscus', 'lateral menisc', 'menisco lateral', 'menisco externo', 'menisque lateral', 'menisque externe', 'laterale meniscus', 'buitenmeniscus', 'aussenmeniskus', 'lateralen? meniskus', 'aussenmeniskushinterhorn', 'lateral menisk', '\\bdis menisk', 'lateralni meniskus', 'lateralnog meniskusa', 'lateralnom meniskusu', 'lateraln\\w* menisk\\w*', '\\blateralnog meniska\\b', 'εξω μηνισκ', 'μηνισκ[^ ]* του εξω', 'εξω διαμερισμα[^.]{0,40}μηνισκ', 'латералния менискус', 'латерален менискус', 'външния менискус', 'oba meniska', 'both menisci', 'ambos meniscos', 'beide menisci', 'her iki menisku', 'αμφοτερ\\w* μηνισκ', 'двата менискуса', 'medial (and|&) lateral menisc')}
OA_EVIDENCE = _rx('osteoarthrit', '\\barthros', '\\bgonarthros', '\\bosteoarthros', 'chondropath', 'chondromalac', 'condropat', 'condromalac', '\\bchondros', '\\bchondrosis\\b', 'chondral (loss|defect|ulcer|thinning|injury|fissur|wear)', 'cartilage (loss|thinning|defect|fissur|wear|damage|heterogeneity|irregularit)', '(loss|thinning|fissur|defect|ulcer|erosion|denudation) of[^.]{0,20}cartilage', 'articular cartilage[^.]{0,30}(loss|thin|fissur|defect|erosion|wear|irregular)', 'osteophyt', 'osteofit', 'osteofyt', 'osteofito', 'osteophyten', 'spurring', 'joint space narrowing', 'pinzamiento articular', 'reduced joint space', 'kikirdak kayb', 'kikirdak incelme', 'kondropati', 'kondral', 'kikirdak dejener', 'eklem aralig\\w* daral', 'eklem mesafesi daral', 'kikirdak kalinlig\\w* azal', 'kraakbeen', 'gonartrose', 'artrose', '\\bknorpel', 'arthrose', 'gonarthrose', 'hrskavic', 'hondromalac', 'artroz', 'osteoartrit', 'artrotsk', 'artrotick', '\\boa promjen', '\\boa\\b', 'degenerativne promjene hrskav', 'χονδρ[^ ]*παθ', 'αρθριτ', 'αρθρωσ', 'οστεοφυτ', 'χονδρομαλακ', 'αρθρικου χονδρου', 'εξαλειψη του αρθρικου χονδρου', 'διαβρωση του αρθρικου χονδρ', 'λεπτυνση[^.]{0,30}χονδρ', 'φθορα[^.]{0,20}χονδρ', 'артроз', 'хондропат', 'остеофит', 'хрущял[^.]{0,40}(изтън|увред|дефект|липс)', 'изтъняване[^.]{0,30}хрущял', 'хондромалац', 'ulcera[s]? condral', 'cartilago[^.]{0,25}(perdida|adelgaz)', 'icrs grade', 'icrs\\b', 'outerbridge', '\\bdenudation\\b', 'denudacij', 'erozivne promjene', '\\berosion of[^.]{0,20}cartilage', 'kraakbeenlijden', 'kraakbeenverlies')
TF_SITE = _rx('compartment', 'compartimento', 'compartiment', 'kompartman', 'kompartiment', 'kompartment', 'odjelj', 'διαμερισμα', 'компартм', '\\bотдел', 'femorotibial', 'tibiofemoral', 'femoro tibial', 'femorotibiaal', 'femorotibijaln', 'феморотибиал', '\\bft zglob', 'tibiofemoraln', 'condyle', 'condilo', 'kondyl', 'kondil', 'condyl', 'κονδυλ', 'кондил', '\\bplateau', '\\bplato\\b', 'platillo', 'meseta', 'плато', 'tibiaplateau', 'tibijaln\\w* plato', 'tibyal plato', 'tibia plato', 'κνημιαι', 'μηριαι', 'weightbearing', 'weightbaring', 'zona de carga', 'dragende deel', 'agirlik tasiyan', '\\bfemur\\b', '\\btibia\\b', '\\bfemoral\\b', '\\btibial\\b', '\\bfemura\\b', '\\btibije\\b', '\\bmesarthrio\\b', 'μεσαρθριο')
PF_SITE = _rx('patellofemoral', 'femoropatellar', 'femoropatelar', 'patelofemoral', 'retropatellar', 'retrorotulian', 'trochlea', 'troclea', 'troklea', 'trochlear', 'trohlej', 'τροχιλ', '\\bpatella', '\\bpatellar', 'rotulian', '\\brotula\\b', '\\bpatele\\b', 'patellofemoraal', 'femoropatellair', 'επιγονατιδ', 'μηροεπιγονατιδ', 'пател', 'феморопател', 'anterior compartment', 'compartimento anterior', 'prednj\\w* odjeljk', '\\bfp zglob', '\\bpf zglob', '\\bfaset', '\\bfacet', 'patellofemoraln')
SIDE_MEDIAL = _rx('\\bmedial\\w*', '\\bmedyal\\w*', '\\bmedijaln\\w*', '\\bmediaal\\w*', '\\bmediale\\w*', '\\binterno\\b', '\\binterna\\b', '\\binternos\\b', '\\binterne\\b', '\\binnen\\w*', '\\bic\\b', '\\bunutarnj\\w*', '\\bεσω\\w*', '\\bεσωτερικ\\w*', '\\bмедиал\\w*', '\\bвътреш\\w*', '\\bbinnen\\w*', '\\bmediaal\\b', '\\bmediales?\\b')
SIDE_LATERAL = _rx('\\blateral\\w*', '\\bexterno\\b', '\\bexterna\\b', '\\bexternos\\b', '\\bexterne\\b', '\\bdis\\b', '\\blateraln\\w*', '\\baussen\\w*', '\\bbuiten\\w*', '\\bεξω\\w*', '\\bεξωτερικ\\w*', '\\bлатерал\\w*', '\\bвъншн\\w*', '\\bvanjsk\\w*')
SIDE_ANTERIOR = _rx('\\banterior\\w*', '\\bant\\b', '\\bon\\b', '\\bprednj\\w*', '\\bvorder\\w*', '\\bvoorste\\b', '\\bπροσθι\\w*', '\\bпредн\\w*', '\\banteriyor\\w*', '\\bavant\\b', '\\banterieur\\w*')
GLOBAL_OA = _rx('tri ?compartment', 'all three compartment', 'global(ised)? (oa|osteoarthrit)', '\\bgonarthros', '\\bgonartros', '\\bgonarthrose', '\\bgonartrose', 'gonartro', 'goanrtrot', 'gonartrot', 'osteoarthritis of the knee', 'artrosis (de |)(la )?rodilla', 'knee osteoarthrit', '\\bdiz osteoartrit', '\\bgonartroz', 'artroza koljena', 'οστεοαρθριτιδα', 'αρθριτιδα του γονατος', 'εκφυλιστικη οστεοαρθριτ', 'артроза на колянната', 'гонартроз', 'degenerative joint disease', '\\bdjd\\b', 'three compartments', 'compartmens', 'compartments')
DIRECT = {'Effusion': _rx('\\beffusion', 'joint fluid', 'intra ?articular fluid', '\\bhydrops\\b', '\\bhemarthros', '\\bhaemarthros', 'derrame articular', '\\bderrame\\b', 'liquido articular', 'hemartrosis', 'epanchement', 'gewrichtsvocht', '\\bvocht\\b', 'gewrichtseffusie', 'opzetting van suprapatell', 'gelenkerguss', '\\berguss\\b', 'gelenksergu', 'gelenksflussigkeit', 'eklem\\w* ic\\w* sivi', 'efuzyon', 'eklem sivisi', 'eklem mesafesinde sivi', 'sivi (miktari|artisi|birikimi)', 'sivi artis', '\\bsivi\\b[^.]{0,25}artmis', '\\bizljev', '\\bizliv', 'zglobn[^ ]* tekucin', '\\bhidrops\\b', 'αρθρικ[^ ]* υγρ', 'υγρου ενδαρθρικα', 'ενδαρθρικ[^ ]* υγρ', 'ποσοτητα υγρου', 'ενδαρθρικ', 'αρθρικη συλλογη', 'υγρο στην αρθρωση', 'υγρου στην αρθρωση', 'συλλογη υγρου', 'ενθαρθρικ', 'ставен излив', 'излив', 'ставна течност', 'синовиална течност'), 'Synovitis': _rx('synovit', 'sinovit', 'synovial (thickening|proliferation|hypertroph)', 'thicken\\w* synovial', 'hypertroph\\w* of the synovium', 'synoviale? (verdikking|proliferatie)', 'verdikkingen van (het )?synovium', 'synovialitis', 'synovialis(verdickung|proliferation)', 'reizsynovial', 'sinovijalitis', 'sinovitis', 'zadebljanje sinovij', 'proliferacij\\w* sinovij', 'sinovijaln\\w* proliferacij', 'υμενιτιδα', 'συνοβιτιδα', 'υμενικ[^ ]* υπερτροφ', 'αρθρικου υμεν', 'παχυνση[^.]{0,20}υμεν', 'υμενα', 'синовит', 'синовиал[^ ]* (задебел|пролифер)', '\\bpannus\\b', '\\bhoffit', 'sinovyal\\w* (kalinlas|proliferas)', 'sinovyal hipertrof', '\\bartrit\\b', '\\barthritis\\b'), "Baker's": _rx('baker', 'popliteal cyst', 'quiste popliteo', 'quistes popliteos', 'kyste poplite', 'popliteale? cyst', 'poplitealzyste', 'bakerzyste', 'popliteal kist', '\\bbakerova\\b', 'poplitealn[^ ]* cist', 'popliteal\\w* cist', 'κυστη baker', 'πολυχωρη συνοβιακη κυστη', 'κυστη του baker', 'συνοβιακη κυστη', 'κυστη τυπου baker', 'киста на бейкър', 'бейкърова киста', 'поплитеална киста', 'бекеров', 'gastrocnemio ?semimembranos', 'gastrocnemius semimembranosus burs'), 'Contusion': _rx('\\bcontusion', 'bone bruise', 'bone marrow (o?edema|contusion)', 'marrow o?edema', '\\bkontuz', 'medular bone o?edema', 'osseous contusion', 'contusion osea', 'edema oseo', 'edema de medula osea', 'contusiones oseas', 'oedeme osseux', 'contusion osseuse', 'botcontusie', 'botoedeem', 'beenmergoedeem', 'botmergoedeem', 'knochenmarkodem', 'knochenodem', 'knochenmarksodem', 'kontusion', 'kemik kontuzyonu', 'kemik iligi odemi', 'kemik odemi', 'kemik iliginde odem', 'kontuzyonel kemik', 'kemik iligi odemleri', 'kostani edem', 'edem kosti', 'kontuzij', 'kostane srzi[^.]{0,20}edem', 'οστεομυελικ[^ ]* οιδημα', 'οστικο οιδημα', 'μυελικο οιδημα', 'οστικο μωλωπ', 'костномозъчен едем', 'костен едем', 'контузионен', 'костно мозъчен едем'), 'Fracture': _rx('\\bfractur', '\\bfract\\b', '\\bfractura', '\\bfracturas\\b', '\\bfractuur', '\\bbreuk\\b', '\\bfraktur', '\\bbruch\\b', '\\bkirik\\b', '\\bkirigi\\b', '\\bkiri[kg]\\w*', '\\bprijelom', 'impresijsk[^ ]* fraktur', 'impaktcij', 'καταγμα', 'καταγματ', 'фрактур', 'счупван', 'фисур', 'insufficiency fracture', 'stress fracture', 'avulsion fracture', 'subchondral fracture', 'subkondral kiri', 'impaction (fracture|injury)', 'osteochondral (fracture|impaction)', '\\bsegond\\b', 'impactiefractuur', 'subchondrale impression', 'subchondraler? impress')}
DECOY = {'Fracture': _rx('microfractur', '\\bfracture (risk|prophyla)'), "Baker's": _rx('meniscal cyst', 'quiste meniscal', 'parameniscal')}
PAIRED = {'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus'}
OA_TARGETS = ['Medial OA', 'Lateral OA', 'PF OA']
PLURAL_MENISCI = _rx('\\bmenisci\\b', '\\bmeniscos\\b', '\\bmenisques\\b', '\\bmenisken\\b', '\\bmeniskusi\\b', '\\bmenisk\\w*ler\\b', '\\bμηνισκοι\\b', '\\bμηνισκων\\b', '\\bменискуси\\b', '\\bменискусите\\b', '\\bmenisci\\w*\\b')
ANY_SIDE = _rx(SIDE_MEDIAL.pattern, SIDE_LATERAL.pattern)
STEM_MENISCUS = _rx('menisc\\w*', 'menisk\\w*', 'μηνισκ\\w*', 'мениск\\w*')
STEM_CRUCIATE = _rx('cruciate', 'cruzado', 'croise', 'kruisband', 'kreuzband', 'capraz bag\\w*', 'krizn\\w*', 'χιαστ\\w*', 'кръстн\\w*', '\\bacl\\b', '\\blca\\b', '\\bvkb\\b', '\\bocb\\b', '\\bacb\\b')
STEM_COLLATERAL = _rx('collateral\\w*', 'colateral\\w*', 'kollateral\\w*', 'collaterale\\w*', 'kolateraln\\w*', 'yan bag\\w*', 'πλαγι\\w*', 'колатерал\\w*', 'странич\\w*', 'innenband\\w*', 'binnenband\\w*', '\\bmcl\\b', '\\blcm\\b', '\\biyb\\b')
STEM_FRACTURE = _rx('fractur\\w*', 'fraktur\\w*', 'fractuur\\w*', '\\bfract\\b', 'kiri[kgğ]\\w*', 'prijelom\\w*', 'lom kosti', '\\bbreuk\\w*', '\\bbruch\\w*', 'καταγμα\\w*', 'καταγματ\\w*', 'фрактур\\w*', 'счупван\\w*', 'fisur\\w* (osea|oseas|kost)', 'fissur\\w* kost')
POSTERIOR_ONLY = _rx('\\bpcl\\b', '\\blcp\\b', '\\bhkb\\b', '\\bacb\\b', 'posterior cruciate', 'cruzado posterior', 'croise posterieur', 'achterste kruisband', 'hinteres kreuzband', 'arka capraz', 'straznji krizn', 'οπισθι[οα]\\w* χιαστ', 'задна кръстн', 'задната кръстн')
LATERAL_COLL_ONLY = _rx('\\blcl\\b', '\\bfcl\\b', 'lateral collateral', 'fibular collateral', 'colateral lateral', 'colateral externo', 'buitenband', 'aussenband', 'dis yan bag', 'lateralni kolateraln', 'εξω πλαγι', 'латерален колатерал')

In [ ]:
def _near(clause: str, stem_rx: re.Pattern, qual_rx: re.Pattern, window: int=55):
    for m in stem_rx.finditer(clause):
        lo = max(0, m.start() - window)
        hi = min(len(clause), m.end() + window)
        if qual_rx.search(clause[lo:hi]):
            return True
    return False
STEM_RULES = {'ACL': (STEM_CRUCIATE, SIDE_ANTERIOR), 'MCL': (STEM_COLLATERAL, SIDE_MEDIAL), 'Medial Meniscus': (STEM_MENISCUS, SIDE_MEDIAL), 'Lateral Meniscus': (STEM_MENISCUS, SIDE_LATERAL)}

class _Matcher:

    def __init__(self, phrase_rx, stem=None, side=None, window=55):
        self.phrase_rx = phrase_rx
        self.stem = stem
        self.side = side
        self.window = window

    def search(self, clause):
        m = self.phrase_rx.search(clause)
        if m is not None:
            return m
        if self.stem is not None and _near(clause, self.stem, self.side, self.window):
            return self.stem.search(clause)
        return None
ANAT_MATCH = {t: _Matcher(ANAT[t], *STEM_RULES[t]) for t in PAIRED}
DIRECT_MATCH = {t: _Matcher(_rx(rx.pattern, STEM_FRACTURE.pattern) if t == 'Fracture' else rx) for t, rx in DIRECT.items()}
SEV_LOW = _rx('\\bsmall\\b', '\\bminimal\\b', '\\btrace\\b', '\\bmild\\b', '\\bslight\\b', '\\btiny\\b', '\\bscant\\b', '\\bdiscrete\\b', '\\blow ?grade\\b', '\\bincipient\\b', '\\bleve\\b', '\\bminim', '\\bpeque', '\\bfina\\b', '\\bfino\\b', '\\bligero\\b', '\\bescaso\\b', '\\bdiscreto\\b', '\\bhafif\\b', '\\baz miktarda\\b', '\\bsilik\\b', '\\bmanj\\w*', '\\bblago\\b', '\\bdiskretn', '\\bmalo\\b', '\\bpocetn', '\\bgering', '\\bdiskret', '\\bkleine?r?\\b', '\\bwenig\\b', '\\bzarte?\\b', '\\bbeperkte?\\b', '\\bgeringe\\b', '\\bweinig\\b', '\\blichte?\\b', '\\blicht\\b', '\\bηπι', '\\bμικρ', '\\bελαχιστ', '\\bαρχομεν', '\\bминимал', '\\bлек', '\\bмалк', '\\bнеголям')
SEV_HIGH = _rx('\\blarge\\b', '\\bmarked\\b', '\\bmassive\\b', '\\bsevere\\b', '\\bextensive\\b', '\\bmoderate\\b', '\\bgross\\b', '\\bsignificant\\b', '\\babundant\\b', '\\btense\\b', '\\bcomplete\\b', '\\bfull ?thickness\\b', '\\bhigh ?grade\\b', '\\badvanced\\b', '\\bmoderad', '\\bimportante\\b', '\\bsevera?\\b', '\\bmarcad', '\\bcuantios', '\\bespesor total\\b', '\\bcompleta?\\b', '\\bbelirgin\\b', '\\byaygin\\b', '\\bileri\\b', '\\bciddi\\b', '\\bbol\\b', '\\bkomplet', '\\bopsezan\\b', '\\bveliki\\b', '\\bizrazit', '\\bznacajn', '\\bumjeren', '\\buznapredoval', '\\bpotpun', '\\bkompleksn', '\\bausgepragt', '\\bdeutlich', '\\bmassiv', '\\bmassig', '\\bgross', '\\buitgebreid', '\\bgevorderd', '\\bveel\\b', '\\bmatige?\\b', '\\bvolledig', '\\bμετρι', '\\bμεγαλ', '\\bεκτεταμεν', '\\bευμεγεθ', '\\bσοβαρ', '\\bπληρη', '\\bголям', '\\bизразен', '\\bзначим', '\\bумерен', '\\bобилен', '\\bпълн')
GRADE_HIGH = re.compile('grade?[ao]?\\s*(3|4|iii|iv)\\b|icrs grade (iii|iv|3|4)|stupnja iv|stupnja iii|\\bgrado (3|4)\\b|\\bgrad (3|4)\\b|\\bgrade (3|4)\\b')
DEGENERATIVE_MARROW = _rx('subchondral', 'subcondral', 'subkondral', 'supkondraln', 'subchondraln', 'υποχονδρι', 'υπαρθρικ', 'субхондрал', 'subchondrale?', 'subartikuler', '\\bcyst', '\\bquist', '\\bzyste\\b', '\\bcistic', 'reactive', 'reactivo', 'degenerative', 'degenerativ', 'reaktiv', '\\bcisti\\b')
TRAUMA = _rx('\\bbruise\\b', '\\bcontusion', '\\bkontuz', '\\btrauma', '\\bimpaction\\b', '\\bpivot shift\\b', '\\bkissing\\b', '\\bacute\\b', '\\bagudo\\b', '\\bakut', '\\bpivot kaymasi\\b', '\\bcontusion osseuse\\b', '\\bbone bruise\\b', '\\bbotcontusie\\b', '\\bконтузион', '\\bμωλωπ', '\\bkontuzij', '\\bimpaktcij', '\\bimpakcij', '\\bfall\\b', '\\binjury\\b', '\\bimpression\\b')
SYNOVIAL_PROXY = _rx('bursit', 'burzit', '\\bbursa\\b[^.]{0,30}(fluid|distend|sivi|tekucin|opzetting)', 'suprapatellar (bursitis|effusion|recess)', 'suprapatellar bursa', 'suprapatellar bursada', 'suprapatelarno', 'suprapatellaire recessus', 'hoffa', 'hoffit', 'plica', 'plika', 'πλικα', 'fat pad[^.]{0,20}(edema|oedema)', 'kapsul', 'capsul', 'καψ', 'капсул', '\\bpannus\\b', '\\bsinov', '\\bsynov')

def _polarity(clause: str, span=None) -> str:
    if UNCERTAIN.search(clause):
        return 'uncertain'
    if span is None or not FEATURES['directional_negation']:
        if NEGATION.search(clause):
            return 'negative'
    elif _negated(clause, span[0], span[1]):
        return 'negative'
    if NORMALITY.search(clause):
        if TEAR.search(clause) or GRADE_HIGH.search(clause):
            return 'positive'
        return 'negative'
    return 'positive'

def _severity(clause: str) -> float:
    high = SEV_HIGH.search(clause) is not None
    low = SEV_LOW.search(clause) is not None
    if high and (not low):
        return 1.0
    if low and (not high):
        return 0.45
    if high and low:
        return 0.8
    return 0.75

def _grade(n_pos, n_neg, n_unc, best):
    if n_pos or n_unc:
        score = min(0.97, 0.5 + 0.45 * best + 0.015 * min(n_pos, 3))
        conf = min(1.0, 0.55 + 0.15 * n_pos)
    elif n_neg:
        score = max(0.04, 0.2 - 0.04 * n_neg)
        conf = min(0.9, 0.45 + 0.12 * n_neg)
    else:
        score, conf = (0.28, 0.05)
    return (score, conf)

def _paired_weight(clause: str, meniscus: bool) -> float:
    g = _grade_of(clause) if FEATURES['graded_pathology'] else None
    tear = TEAR.search(clause) is not None
    if meniscus:
        if tear:
            base = 1.0
        elif g is not None:
            base = 0.95 if g >= 3 else 0.3
        elif DEGEN.search(clause):
            base = 0.35
        else:
            base = 0.45
    elif tear:
        base = 1.0
    elif g is not None:
        base = 0.85 if g >= 2 else 0.3
    elif DEGEN.search(clause):
        base = 0.4
    else:
        base = 0.55
    if SEV_HIGH.search(clause) and (not SEV_LOW.search(clause)):
        base = min(1.0, base * 1.2)
    elif SEV_LOW.search(clause) and (not SEV_HIGH.search(clause)):
        base *= 0.7
    return base

def _score_paired(cls, tgt):
    anat_rx = ANAT_MATCH[tgt]
    path_rx = _rx(TEAR.pattern, DEGEN.pattern, INJURY.pattern)
    meniscus = 'Meniscus' in tgt
    n_pos = n_neg = n_unc = 0
    best = 0.0
    for c in cls:
        hit = anat_rx.search(c)
        if hit is None and meniscus and PLURAL_MENISCI.search(c) and (not ANY_SIDE.search(c)):
            hit = PLURAL_MENISCI.search(c)
        if hit is None:
            continue
        pm = path_rx.search(c)
        if pm is None and _grade_of(c) is None:
            if NORMAL_PHRASE.search(c) or (NORMALITY.search(c) and (not NEGATION.search(c))):
                n_neg += 1
            continue
        span = (pm.start(), pm.end()) if pm is not None else None
        pol = _polarity(c, span)
        if pol == 'positive':
            n_pos += 1
            best = max(best, _paired_weight(c, meniscus))
        elif pol == 'negative':
            n_neg += 1
        else:
            n_unc += 1
            best = max(best, 0.45 * _paired_weight(c, meniscus))
    s, cf = _grade(n_pos, n_neg, n_unc, best)
    return (s, cf, n_pos, n_neg)

def _score_clauses(cls, anat_rx, path_rx=None, decoy_rx=None, context_penalty=None, context_bonus=None):
    n_pos = n_neg = n_unc = 0
    best = 0.0
    for c in cls:
        m = anat_rx.search(c)
        if not m:
            continue
        if decoy_rx is not None and decoy_rx.search(c):
            continue
        if path_rx is not None and (not path_rx.search(c)):
            if NORMAL_PHRASE.search(c) or (NORMALITY.search(c) and (not NEGATION.search(c))):
                n_neg += 1
            continue
        pol = _polarity(c, (m.start(), m.end()))
        if pol == 'positive':
            n_pos += 1
            w = _severity(c)
            if context_penalty is not None and context_penalty.search(c):
                w *= 0.45
            if context_bonus is not None and context_bonus.search(c):
                w = min(1.0, w * 1.35)
            best = max(best, w)
        elif pol == 'negative':
            n_neg += 1
        else:
            n_unc += 1
            best = max(best, 0.3)
    s, c = _grade(n_pos, n_neg, n_unc, best)
    return (s, c, n_pos, n_neg)

def _score_oa(cls):
    acc = {t: {'pos': 0, 'neg': 0, 'unc': 0, 'best': 0.0} for t in OA_TARGETS}
    g_pos, g_neg, g_best = (0, 0, 0.0)
    for c in cls:
        m = OA_EVIDENCE.search(c)
        if not m:
            continue
        pol = _polarity(c, (m.start(), m.end()))
        sev = _severity(c)
        tf_med = _near(c, TF_SITE, SIDE_MEDIAL, 45)
        tf_lat = _near(c, TF_SITE, SIDE_LATERAL, 45)
        pf = PF_SITE.search(c) is not None
        hits = []
        if tf_med:
            hits.append('Medial OA')
        if tf_lat:
            hits.append('Lateral OA')
        if pf:
            hits.append('PF OA')
        if not hits:
            if pol == 'positive':
                g_pos += 1
                g_best = max(g_best, sev if GLOBAL_OA.search(c) else sev * 0.7)
            elif pol == 'negative':
                g_neg += 1
            continue
        for t in hits:
            if pol == 'positive':
                acc[t]['pos'] += 1
                acc[t]['best'] = max(acc[t]['best'], sev)
            elif pol == 'negative':
                acc[t]['neg'] += 1
            else:
                acc[t]['unc'] += 1
                acc[t]['best'] = max(acc[t]['best'], 0.3)
    out = {}
    for t in OA_TARGETS:
        a = acc[t]
        pos, neg, unc, best = (a['pos'], a['neg'], a['unc'], a['best'])
        if not (pos or unc) and g_pos and FEATURES['oa_inherit']:
            if neg:
                score, conf = _grade(0, neg, 0, 0.0)
                score = max(score, 0.35)
                conf *= 0.7
            else:
                score, conf = _grade(g_pos, 0, 0, g_best * 0.92)
                conf *= 0.75
        else:
            score, conf = _grade(pos, neg + g_neg, unc, best)
        out[t] = (score, conf, pos, neg)
    return out

def extract(report: str) -> dict:
    cls = clauses(report)
    out = {}
    for tgt in PAIRED:
        s, c, npos, nneg = _score_paired(cls, tgt)
        out[tgt] = s
        out[tgt + '__conf'] = c
        out[tgt + '__npos'] = npos
        out[tgt + '__nneg'] = nneg
    for tgt, (s, c, npos, nneg) in _score_oa(cls).items():
        out[tgt] = s
        out[tgt + '__conf'] = c
        out[tgt + '__npos'] = npos
        out[tgt + '__nneg'] = nneg
    for tgt in ('Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture'):
        if tgt == 'Contusion':
            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt), context_penalty=DEGENERATIVE_MARROW, context_bonus=TRAUMA)
        else:
            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt))
        out[tgt] = s
        out[tgt + '__conf'] = c
        out[tgt + '__npos'] = npos
        out[tgt + '__nneg'] = nneg
    if FEATURES['synovitis_backoff'] and out['Synovitis__npos'] == 0 and (out['Synovitis__nneg'] == 0):
        proxy = sum((1 for c in cls if SYNOVIAL_PROXY.search(c) and _polarity(c) == 'positive'))
        eff = out['Effusion']
        prior = 0.3 + 0.3 * max(0.0, (eff - 0.5) / 0.45) + 0.06 * min(proxy, 3)
        out['Synovitis'] = min(0.72, prior)
        out['Synovitis__conf'] = 0.18
    return out

In [ ]:
import os
import time
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
T0 = time.time()

def log(msg):
    print(f'[{time.time() - T0:7.1f}s] {msg}', flush=True)

def find_root():
    for c in [Path('/kaggle/input/rsna-knee-abnormality-detection'), Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'), Path('data'), Path('.')]:
        if (c / 'test.csv').is_file() and (c / 'test_series').is_dir():
            return c
    base = Path('/kaggle/input')
    if base.is_dir():
        for d1 in sorted((p for p in base.iterdir() if p.is_dir())):
            for cand in [d1] + sorted((p for p in d1.iterdir() if p.is_dir())):
                if (cand / 'test.csv').is_file():
                    return cand
    raise FileNotFoundError('competition mount not found')
ROOT = find_root()
log(f'input root: {ROOT}')
_test_df = pd.read_csv(ROOT / 'test.csv')
_bench = _test_df[['StudyInstanceUID']].copy()
for _c in TARGETS:
    _bench[_c] = 0.5
_bench.to_csv('submission.csv', index=False)
log(f'benchmark submission.csv written ({len(_bench)} rows)')
STAGE_OK = {}

def stage(name):

    def deco(fn):

        def run(*a, **k):
            t = time.time()
            try:
                out = fn(*a, **k)
                STAGE_OK[name] = True
                log(f"stage '{name}' ok in {time.time() - t:.1f}s")
                return out
            except Exception:
                import traceback
                traceback.print_exc()
                STAGE_OK[name] = False
                log(f"stage '{name}' FAILED after {time.time() - t:.1f}s")
                return None
        return run
    return deco
train_df = pd.read_csv(ROOT / 'train.csv')
log(f'train {train_df.shape}  test {_test_df.shape}')
t = time.time()
LAB = pd.DataFrame([extract(r) for r in train_df['Report'].fillna('')])
LAB['StudyInstanceUID'] = train_df['StudyInstanceUID'].values
LAB = LAB.set_index('StudyInstanceUID')
log(f'read {len(LAB)} reports in {time.time() - t:.1f}s')
GOLD = train_df.dropna(subset=TARGETS).set_index('StudyInstanceUID')[TARGETS]
log(f'{len(GOLD)} studies carry the twelve annotations')
pos = (LAB[TARGETS] > 0.5).mean()
sil = pd.Series({t_: float(((LAB[t_ + '__npos'] == 0) & (LAB[t_ + '__nneg'] == 0)).mean()) for t_ in TARGETS})
print(pd.DataFrame({'derived positive rate': pos.round(3), 'silence rate': sil.round(3), 'annotated positive rate': GOLD.mean().round(3)}).to_string())

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
plt.rcParams.update({'figure.dpi': 120, 'font.size': 8, 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})
INK, ACC, WARN = ('#22303f', '#2b7a9b', '#c25a3d')

def agreement(lab, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    g = lab.loc[GOLD.index]
    rows = []
    for t_ in TARGETS:
        y = GOLD[t_].values.astype(int)
        p = g[t_].values
        if len(set(y)) < 2:
            rows.append((t_, np.nan, np.nan, np.nan, int(y.sum()), int((1 - y).sum())))
            continue
        a = roc_auc_score(y, p)
        bs = []
        for _ in range(n_boot):
            i = rng.integers(0, len(y), len(y))
            if len(set(y[i])) > 1:
                bs.append(roc_auc_score(y[i], p[i]))
        rows.append((t_, a, np.percentile(bs, 2.5), np.percentile(bs, 97.5), int(y.sum()), int((1 - y).sum())))
    return pd.DataFrame(rows, columns=['target', 'auc', 'lo', 'hi', 'npos', 'nneg'])
AGREE = agreement(LAB).dropna(subset=['auc'])
print(AGREE.round(3).to_string(index=False))
print(f'\nmacro agreement AUC: {AGREE.auc.mean():.4f}   mean silence rate: {sil.mean() * 100:.1f}%')
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.6), gridspec_kw={'width_ratios': [1.25, 1]})
o = AGREE.sort_values('auc')
y = np.arange(len(o))
ax[0].hlines(y, o.lo, o.hi, color=ACC, lw=3, alpha=0.35)
ax[0].plot(o.auc, y, 'o', color=ACC, ms=5)
ax[0].axvline(0.5, color=INK, lw=0.8, ls=':')
ax[0].axvline(o.auc.mean(), color=WARN, lw=1, ls='--')
ax[0].text(o.auc.mean(), -0.9, f' macro {o.auc.mean():.3f}', color=WARN, fontsize=7)
ax[0].set_ylim(-1.4, len(o) - 0.4)
ax[0].set_yticks(y)
ax[0].set_yticklabels([f'{t_}  ({p}+/{n}-)' for t_, p, n in zip(o.target, o.npos, o.nneg)])
ax[0].set_xlim(0.35, 1.02)
ax[0].set_xlabel('AUC of the derived score against the annotation')
ax[0].set_title('gauge one: agreement, n = 58\nbars are 95% bootstrap intervals — they are this wide on purpose', loc='left', fontsize=8)
o2 = sil.sort_values()
ax[1].barh(np.arange(len(o2)), o2.values * 100, color=INK, alpha=0.8, height=0.65)
ax[1].set_yticks(np.arange(len(o2)))
ax[1].set_yticklabels(o2.index)
ax[1].set_xlabel('% of studies where no rule fired at all')
ax[1].set_title('gauge two: coverage, n = 4 407\nno labels needed, so it runs on the whole corpus', loc='left', fontsize=8)
for i, v in enumerate(o2.values * 100):
    ax[1].text(v + 1, i, f'{v:.0f}', va='center', fontsize=6.5, color=INK)
fig.tight_layout()
plt.show()

In [ ]:
_SCRIPT = {'el': re.compile('[Ͱ-Ͽ]'), 'bg/ru': re.compile('[Ѐ-ӿ]')}
_STOP = {'en': '\\b(the|and|is|with|there is|normal)\\b', 'es': '\\b(del|los|las|con|sin|senal|rodilla|hallazgos|tecnica|resultados|impresion|menisco|rotura)\\b', 'fr': '\\b(des|les|avec|sans|genou|aucune)\\b', 'nl': '\\b(van|het|een|geen|met|voorste|knie)\\b', 'de': '\\b(der|die|und|mit|ohne|kein|keine|nachweis)\\b', 'tr': '\\b(ve|ile|izlenmistir|mevcut|normaldir|diz|bulgular)\\b', 'hr': '\\b(se|te|uz|bez|prikaz|uredan|koljena|meniska)\\b'}
_STOP = {k: re.compile(v) for k, v in _STOP.items()}

def guess_language(report):
    n = normalize(report)
    for tag, rx in _SCRIPT.items():
        if rx.search(n):
            return tag
    score = {k: len(rx.findall(n)) for k, rx in _STOP.items()}
    best = max(score, key=score.get)
    return best if score[best] >= 2 else '?'
LANG = pd.Series([guess_language(r) for r in train_df['Report'].fillna('')], index=train_df['StudyInstanceUID'])
print(LANG.value_counts().to_string())
SIL = pd.DataFrame({t: ((LAB[t + '__npos'] == 0) & (LAB[t + '__nneg'] == 0)).values for t in TARGETS}, index=LAB.index)
by_lang = SIL.groupby(LANG.reindex(SIL.index).values).mean() * 100
by_lang = by_lang.loc[LANG.value_counts().index.intersection(by_lang.index)]
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw={'width_ratios': [1.7, 1]})
im = ax[0].imshow(by_lang.values, cmap='RdYlBu_r', vmin=0, vmax=100, aspect='auto')
ax[0].set_xticks(range(len(TARGETS)))
ax[0].set_xticklabels(TARGETS, rotation=55, ha='right', fontsize=6.5)
ax[0].set_yticks(range(len(by_lang)))
ax[0].set_yticklabels([f'{l}  (n={int((LANG == l).sum())})' for l in by_lang.index], fontsize=7)
ax[0].grid(False)
for i in range(by_lang.shape[0]):
    for j in range(by_lang.shape[1]):
        v = by_lang.values[i, j]
        ax[0].text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=5.5, color='white' if v > 62 or v < 12 else INK)
ax[0].set_title('gauge two, broken down: % of studies where no rule fired\na common finding silent in one language and not another is a lexicon gap — or a reporting style', loc='left', fontsize=8)
fig.colorbar(im, ax=ax[0], fraction=0.02, pad=0.01)
CLAUSES = {u: clauses(r) for u, r in zip(train_df['StudyInstanceUID'], train_df['Report'].fillna(''))}

def names_it(uid, target):
    cs = CLAUSES[uid]
    if any((ANAT_MATCH[target].search(c) for c in cs)):
        return True
    if 'Meniscus' in target:
        return any((PLURAL_MENISCI.search(c) and (not ANY_SIDE.search(c)) for c in cs))
    return False
rows = []
for t_ in ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus']:
    sel = SIL[t_].values
    if not sel.sum():
        continue
    named = np.array([names_it(u, t_) for u in SIL.index[sel]])
    rows.append((t_, int(sel.sum()), 100 * named.mean()))
D = pd.DataFrame(rows, columns=['target', 'silent', 'names it anyway'])
y = np.arange(len(D))
ax[1].barh(y, 100 - D['names it anyway'], color='#8a97a3', label='never mentioned')
ax[1].barh(y, D['names it anyway'], left=100 - D['names it anyway'], color=WARN, label='mentioned, missed')
ax[1].set_yticks(y)
ax[1].set_yticklabels([f'{t_}\n({n} silent)' for t_, n in zip(D.target, D.silent)], fontsize=6.5)
ax[1].set_xlabel('% of the silent studies')
ax[1].legend(fontsize=6.5, frameon=False, loc='lower right')
ax[1].set_title('why it was silent\nonly the orange half is a lexicon gap', loc='left', fontsize=8)
fig.tight_layout()
plt.show()
print(D.round(1).to_string(index=False))

In [ ]:
from __future__ import annotations
import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_v, '4')
import gc
import hashlib
import json
import re
import time
import traceback
import threading
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F

def _cuda_execution_probe(index):
    dev = torch.device(f'cuda:{index}')
    try:
        major, minor = torch.cuda.get_device_capability(index)
        probe = nn.Conv2d(3, 4, kernel_size=3, padding=1).eval().to(dev)
        with torch.inference_mode():
            out = probe(torch.zeros((1, 3, 16, 16), device=dev))
            if tuple(out.shape) != (1, 4, 16, 16):
                raise RuntimeError(f'unexpected CUDA probe shape {tuple(out.shape)}')
        torch.cuda.synchronize(index)
        print(f'cuda:{index} probe PASS (compute {major}.{minor})')
        del probe, out
        torch.cuda.empty_cache()
        return True
    except Exception as exc:
        print(f'cuda:{index} probe FAIL ({type(exc).__name__}: {exc}); using CPU fallback')
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        return False
DEVS = []
if torch.cuda.is_available():
    DEVS = [torch.device(f'cuda:{i}') for i in range(torch.cuda.device_count()) if _cuda_execution_probe(i)]
if not DEVS:
    DEVS = [torch.device('cpu')]
print(f'devices: {[str(d) for d in DEVS]}')
T0 = time.time()
SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
CROP_MM = 130.0
CACHE_IMG = 336
GROUP = 3
N_GROUP_MAX = 1
CACHE_FRACTION = 0.45
CACHE_BUDGET_MAX_GB = 24.0
CACHE_BUDGET_GB = 12.0
TEST_SHARE = 0.3
HDR_THREADS = 16
PIX_THREADS = 12
ORDER_THREADS = 32
ORDER_BUDGET_S = 5400
RUNS = [{'name': 'r224', 'img': 224}, {'name': 'r336', 'img': 336}]
EPOCHS = 10
BATCH_STUDIES = 8
AUG_ROT_DEG = 8.0
AUG_SCALE = 0.08
AUG_SHIFT = 0.05
AUG_INTENSITY = 0.1
LAT_MIN_OFFSET_MM = 20.0
SLICE_BAND = (0.2, 0.8)
RULES_NATIVE = {'order': 'normal', 'lat': 'centre', 'slot_fallback': False, 'decode_fill': 'nearest'}
RULES_LEGACY = {'order': 'dominant_axis', 'lat': 'corner_x', 'slot_fallback': True, 'decode_fill': 'zero'}
RULES = dict(RULES_NATIVE)
LEGACY_LAT_OFFSET_MM = 5.0
LR_HEAD = 0.001
LR_BACKBONE = 8e-06
UNFREEZE_LAST = 6
WEIGHT_DECAY = 0.02
EVAL_BATCH = 8
TIME_BUDGET = 8.0 * 3600
SLOTS_RECOVERED = [('SAG_FLUID_FS', 'Sagittal', True, True), ('COR_FLUID_FS', 'Coronal', True, True), ('AX_FLUID_FS', 'Axial', True, True), ('SAG_FLUID_NOFS', 'Sagittal', True, False), ('COR_T1', 'Coronal', False, False), ('SAG_T1', 'Sagittal', False, False)]
SLOTS_PUBLIC = [('SAG_FLUID', 'Sagittal', None, True), ('COR_FLUID', 'Coronal', None, True), ('AX_FLUID', 'Axial', None, True), ('SAG_STRUCT', 'Sagittal', None, False), ('COR_STRUCT', 'Coronal', None, False), ('AX_STRUCT', 'Axial', None, False)]
SLOT_SCHEME = os.environ.get('SLOT_SCHEME', 'recovered')
SLOTS = SLOTS_PUBLIC if SLOT_SCHEME == 'public' else SLOTS_RECOVERED
N_SLOT = len(SLOTS)
POOL_PARTS = {'cls_mean': 2, 'cls_mean_focal': 3}
SLOT_PRIOR_TABLE = {'ACL': (0, 3, 5), 'MCL': (1, 4), 'Medial Meniscus': (0, 1, 3, 4), 'Lateral Meniscus': (0, 1, 3, 4), 'Medial OA': (1, 4, 5), 'Lateral OA': (1, 4, 5), 'PF OA': (0, 2, 5), 'Effusion': (0, 2), 'Synovitis': (0, 2), "Baker's": (0,), 'Contusion': (0, 1, 2), 'Fracture': (0, 1, 2, 4, 5)}
SLOT_PRIOR_STRENGTH = 0.55
FATSAT_OPTS = {'FS', 'FATSAT', 'FAT_SAT', 'FSAT'}
_SEP = re.compile('[_\\-.]')
_FATSAT_RX = re.compile('\\bfs\\b|fatsat|fat sat|\\bstir\\b|\\bspair\\b|\\bspir\\b|\\bwe\\b|water excit|\\btirm\\b|\\bsting\\b|\\bfatsup\\b')
_T1_RX = re.compile('\\bt1\\b|\\bt1w\\b')
_T2_RX = re.compile('\\bt2\\b|\\bt2w\\b')
_PD_RX = re.compile('\\bpd\\b|\\bpdw\\b|proton|\\bdp\\b|dens')

In [ ]:
def log(msg):
    print(f'[{time.time() - T0:7.1f}s] {msg}', flush=True)

def find_root():
    for c in [Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'), Path('/kaggle/input/rsna-knee-abnormality-detection'), Path('data'), Path('.')]:
        if (c / 'test.csv').is_file() and (c / 'test_series').is_dir():
            return c
    base = Path('/kaggle/input')
    if base.is_dir():
        for depth1 in sorted((p for p in base.iterdir() if p.is_dir())):
            for cand in [depth1] + sorted((p for p in depth1.iterdir() if p.is_dir())):
                if (cand / 'test.csv').is_file():
                    return cand
    raise FileNotFoundError(f'competition mount not found (cwd {Path.cwd()}); expected a directory holding test.csv and test_series/')

def find_dinov2(variant='small'):
    base = Path('/kaggle/input')
    if not base.is_dir():
        return None
    hits = []
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series')]
        if 'config.json' in files and 'dinov2' in root.lower():
            hits.append(Path(root))
    for h in hits:
        if variant in str(h).lower():
            return h
    return hits[0] if hits else None
LABEL_COLS = TARGETS + [t + '__conf' for t in TARGETS]

class LabelSourceError(RuntimeError):
    pass

def find_label_table():
    base = Path('/kaggle/input')
    cands = []
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series')]
            cands += [Path(root) / f for f in files if f.startswith('report_labels') and f.endswith('.csv')]
    cands += [p for p in (Path('data/derived/report_labels_v2.csv'),) if p.is_file()]
    for c in cands:
        try:
            head = pd.read_csv(c, nrows=1)
        except Exception:
            continue
        if 'StudyInstanceUID' in head.columns and all((t in head.columns for t in TARGETS)):
            return c
    return None

def label_mount_attached():
    base = Path('/kaggle/input')
    if not base.is_dir():
        return False
    return any(('label' in p.name.lower() for p in base.iterdir() if p.is_dir()))

def read_labels(train_df):
    n = len(train_df)
    lab = pd.DataFrame([extract(r) for r in train_df['Report'].fillna('')])
    lab['StudyInstanceUID'] = train_df['StudyInstanceUID'].values
    lab = lab.set_index('StudyInstanceUID')
    src = find_label_table()
    if src is None:
        if label_mount_attached():
            raise LabelSourceError('LABEL SOURCE: a label dataset is mounted but no usable table was found in it. Falling back to the lexicon here would train on the weaker labels and say so only in a log line, so the run stops instead.')
        log(f'LABEL SOURCE: lexicon, {n} studies (no table mounted)')
        return lab
    tab = pd.read_csv(src).set_index('StudyInstanceUID')
    missing = [c for c in LABEL_COLS if c not in tab.columns]
    if missing:
        raise LabelSourceError(f'LABEL SOURCE: {src} is missing {len(missing)} expected columns (first: {missing[0]!r}). Refusing to fall back silently.')
    hit = lab.index.intersection(tab.index)
    if not len(hit):
        raise LabelSourceError(f'LABEL SOURCE: {src} shares no StudyInstanceUID with train.csv.')
    log(f'LABEL SOURCE: {src.name} covers {len(hit)} of {n} studies, lexicon for the remaining {n - len(hit)}')
    lab.loc[hit, LABEL_COLS] = tab.loc[hit, LABEL_COLS].values
    return lab
ROOT = find_root()
log(f'input root: {ROOT}')
IMG = CACHE_IMG

def available_gb():
    try:
        with open('/proc/meminfo') as fh:
            info = {k.strip(): v for k, v in (l.split(':', 1) for l in fh if ':' in l)}
        return int(info['MemAvailable'].split()[0]) / 1024 ** 2
    except Exception:
        return CACHE_BUDGET_GB / CACHE_FRACTION

def plan_cache(n_study, n_test=0):
    avail = available_gb()
    budget = min(avail * CACHE_FRACTION, CACHE_BUDGET_MAX_GB)
    n_total = n_study + max(n_test, int(TEST_SHARE * n_study))
    per_slice = n_total * N_SLOT * IMG * IMG
    afford = int(budget * 1024 ** 3 // max(per_slice, 1))
    groups = max(1, min(N_GROUP_MAX, afford // GROUP))
    log(f'memory: {avail:.1f} GB available, {budget:.1f} GB to the cache; sizing for {n_study} train + {n_total - n_study} test studies -> {groups} group(s) of {GROUP} = {groups * GROUP} slices per slot' + (f' (wanted {N_GROUP_MAX})' if groups < N_GROUP_MAX else ''))
    return groups
N_GROUP = plan_cache(len(pd.read_csv(ROOT / 'train.csv')), len(pd.read_csv(ROOT / 'test.csv')))
CACHE_SLICES = GROUP * N_GROUP
log(f'cache layout: {N_GROUP} groups x {GROUP} slices = {CACHE_SLICES} per slot')

In [ ]:
HDR_TAGS = ['SeriesDescription', 'SequenceName', 'ScanOptions', 'ScanningSequence', 'RepetitionTime', 'EchoTime', 'Laterality', 'PixelSpacing', 'Rows', 'Columns', 'RescaleSlope', 'RescaleIntercept', 'ImagePositionPatient', 'ImageOrientationPatient']

def _hdr_vec(s, n):
    if not isinstance(s, str):
        return None
    try:
        v = [float(x) for x in s.split('|')]
    except ValueError:
        return None
    return np.array(v) if len(v) >= n else None

def side_from_geometry(h):
    cx = {}
    for r in h.itertuples(index=False):
        ipp = _hdr_vec(getattr(r, 'ImagePositionPatient', None), 3)
        iop = _hdr_vec(getattr(r, 'ImageOrientationPatient', None), 6)
        ps = _hdr_vec(getattr(r, 'PixelSpacing', None), 2)
        rows, cols = (getattr(r, 'Rows', None), getattr(r, 'Columns', None))
        if ipp is None or iop is None or ps is None or (not rows) or (not cols):
            continue
        try:
            c = ipp[:3] + iop[:3] * ps[1] * float(cols) / 2 + iop[3:6] * ps[0] * float(rows) / 2
        except (TypeError, ValueError):
            continue
        cx.setdefault(r.StudyInstanceUID, []).append(float(c[0]))
    out = {}
    for st, xs in cx.items():
        m = float(np.median(xs))
        out[st] = None if abs(m) < LAT_MIN_OFFSET_MM else 'R' if m < 0 else 'L'
    return out

def side_from_corner_x(h):
    out = {}
    for st, g in h.groupby('StudyInstanceUID'):
        xs = []
        for r in g.itertuples(index=False):
            ipp = _hdr_vec(getattr(r, 'ImagePositionPatient', None), 3)
            if ipp is not None and np.isfinite(ipp).all():
                xs.append(float(ipp[0]))
        if not xs:
            out[st] = None
            continue
        x = float(np.median(xs))
        out[st] = None if abs(x) < LEGACY_LAT_OFFSET_MM else 'R' if x < 0 else 'L'
    return out

def lat_of(h, tag=''):
    geo = side_from_corner_x(h) if RULES['lat'] == 'corner_x' else side_from_geometry(h)
    d, n_tag, n_geo, n_none, n_disagree = ({}, 0, 0, 0, 0)
    for st, g in h.groupby('StudyInstanceUID'):
        v = [str(x).strip().upper() for x in g['Laterality'].dropna()]
        if RULES['lat'] == 'corner_x' and 'ImageLaterality' in g.columns:
            v += [str(x).strip().upper() for x in g['ImageLaterality'].dropna()]
        v = [x[0] for x in v if x and x[0] in ('L', 'R')]
        side = v[0] if v else None
        if side is not None:
            n_tag += 1
            if geo.get(st) is not None and geo[st] != side:
                n_disagree += 1
        else:
            side = geo.get(st)
            n_geo += side is not None
            n_none += side is None
        d[st] = side
    log(f'{tag}laterality: {n_tag} from the tag, {n_geo} from geometry, {n_none} unresolved; tag and geometry disagree on {n_disagree} ({n_disagree / max(n_tag, 1):.1%} of the tagged)')
    return d

def probe(item):
    split, study, series, path = item
    row = {'split': split, 'StudyInstanceUID': study, 'SeriesInstanceUID': series, 'dir': path}
    try:
        files = sorted((e.name for e in os.scandir(path) if e.name.endswith('.dcm')))
        row['files'] = files
        row['n_slices'] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]), stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == 'MultiValue':
                row[t] = '|'.join((str(x) for x in v))
            else:
                row[t] = str(v)
    except Exception as exc:
        row['err'] = str(exc)[:120]
    return row

def walk(split):
    base = ROOT / split
    items = []
    if not base.is_dir():
        return pd.DataFrame(columns=['split', 'StudyInstanceUID', 'SeriesInstanceUID', 'dir', 'files', 'n_slices'] + HDR_TAGS)
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe, items))
    return pd.DataFrame(rows)

def annotate(df):
    desc = df['SeriesDescription'].fillna('') + ' ' + df['SequenceName'].fillna('')
    desc = desc.str.lower().str.replace(_SEP, ' ', regex=True)
    opts = df['ScanOptions'].fillna('').str.upper().str.split('|')
    opts_fs = opts.apply(lambda ts: any((t.strip() in FATSAT_OPTS for t in ts)))
    df['fatsat'] = desc.str.contains(_FATSAT_RX) | opts_fs
    tr = pd.to_numeric(df['RepetitionTime'], errors='coerce')
    te = pd.to_numeric(df['EchoTime'], errors='coerce')
    gre = df['ScanningSequence'].fillna('').str.upper().str.contains('GR')
    t1, t2, pdw = (desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX))
    df['weight'] = np.where(t1 & ~t2 & ~pdw, 'T1', np.where(t2 & ~pdw, 'T2', np.where(pdw, 'PD', np.where(gre, 'GRE', np.where(tr < 800, 'T1', np.where(te > 60, 'T2', np.where(tr >= 800, 'PD', 'UNK')))))))
    df['fluid'] = np.isin(df['weight'], ['PD', 'T2'])
    df['px'] = pd.to_numeric(df['PixelSpacing'].fillna('').str.split('|').str[0].replace('', np.nan), errors='coerce')
    return df

In [ ]:
def pick_slots(series_df, plane_map):
    series_df = series_df.copy()
    series_df['plane'] = series_df['SeriesInstanceUID'].map(plane_map)
    out = {}
    for study, g in series_df.groupby('StudyInstanceUID'):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            sel = (g['plane'] == plane) & (g['fatsat'] == fs)
            if fluid is not None:
                sel &= g['fluid'] == fluid
            cand = g[sel]
            if len(cand) == 0 and RULES['slot_fallback'] and (fluid is False):
                cand = g[(g['plane'] == plane) & ~g['fatsat']]
            if len(cand):
                chosen[name] = cand.sort_values('n_slices', ascending=False).iloc[0]
        out[study] = chosen
    return out

In [ ]:
ORDER_TAGS = [(32, 50), (32, 55), (32, 19)]
DECODE_FAILED = []

def cache_tag(rules=None):
    r = dict(RULES if rules is None else rules)
    t = f'{CACHE_IMG}px_{CACHE_SLICES}sl_{int(CROP_MM)}mm_{SLICE_BAND[0]:.2f}-{SLICE_BAND[1]:.2f}'
    if {k: r.get(k, v) for k, v in RULES_NATIVE.items()} != RULES_NATIVE:
        t += '_' + hashlib.md5(json.dumps(r, sort_keys=True).encode()).hexdigest()[:6]
    return t

def _natural_key(name):
    return tuple((int(x) if x.isdigit() else x.lower() for x in re.split('(\\d+)', str(name))))

def _order_dominant_axis(rec):
    files, d = (rec['files'], rec['dir'])
    rows = []
    for pos, f in enumerate(files):
        ipp = inst = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True, specific_tags=['ImagePositionPatient', 'InstanceNumber'])
            raw = getattr(ds, 'ImagePositionPatient', None)
            if raw is not None and len(raw) >= 3:
                c = np.asarray(raw[:3], dtype=np.float64)
                if np.isfinite(c).all():
                    ipp = c
            n = getattr(ds, 'InstanceNumber', None)
            if n is not None:
                inst = float(n)
        except Exception:
            pass
        rows.append((f, ipp, inst, pos))
    placed = [r for r in rows if r[1] is not None]
    need = max(2, int(0.8 * len(rows)))
    if len(placed) >= need:
        xyz = np.stack([r[1] for r in placed])
        axis = int(np.argmax(np.ptp(xyz, axis=0)))
        spare = float(np.nanmedian(xyz[:, axis]))
        rows.sort(key=lambda r: (float(r[1][axis]) if r[1] is not None else spare, r[2] if r[2] is not None else float('inf'), r[3]))
    elif sum((r[2] is not None for r in rows)) >= need:
        rows.sort(key=lambda r: (r[2] if r[2] is not None else float('inf'), r[3]))
    else:
        rows.sort(key=lambda r: _natural_key(r[0]))
    return ([r[0] for r in rows], True)

def order_slices(rec):
    if RULES['order'] == 'dominant_axis':
        return _order_dominant_axis(rec)
    files, d = (rec['files'], rec['dir'])
    keyed = []
    for f in files:
        k = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True, specific_tags=ORDER_TAGS)
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            k = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
        except Exception:
            try:
                k = float(ds.InstanceNumber)
            except Exception:
                k = None
        keyed.append((k, f))
    if any((k is None for k, _ in keyed)):
        return (files, False)
    return ([f for _, f in sorted(keyed, key=lambda t: t[0])], True)

def read_slot(rec, n_slice=None, out_size=None):
    n_slice = GROUP if n_slice is None else n_slice
    out_size = IMG if out_size is None else out_size
    files, d, px = (rec.get('ordered') or rec['files'], rec['dir'], rec['px'])
    n = len(files)
    if n == 0:
        return None
    lo, hi = (int(SLICE_BAND[0] * (n - 1)), int(SLICE_BAND[1] * (n - 1)))
    idx = np.unique(np.linspace(lo, hi, n_slice).astype(int)) if hi > lo else np.array([n // 2])
    while len(idx) < n_slice:
        idx = np.append(idx, idx[-1])
    planes = []
    for i in idx[:n_slice]:
        try:
            ds = pydicom.dcmread(os.path.join(d, files[int(i)]), force=True)
            a = ds.pixel_array.astype(np.float32)
            sl = float(getattr(ds, 'RescaleSlope', 1) or 1)
            ic = float(getattr(ds, 'RescaleIntercept', 0) or 0)
            a = a * sl + ic
        except Exception:
            a = None
        planes.append(a)
    got = [k for k, p in enumerate(planes) if p is not None]
    if RULES['decode_fill'] == 'zero':
        if not got:
            DECODE_FAILED.append(rec.get('SeriesInstanceUID', d))
        planes = [np.zeros((out_size, out_size), np.float32) if p is None else p for p in planes]
        got = list(range(len(planes)))
    if not got:
        DECODE_FAILED.append(rec.get('SeriesInstanceUID', d))
        return None
    if len(got) < len(planes):
        DECODE_FAILED.append(rec.get('SeriesInstanceUID', d))
        for k, p in enumerate(planes):
            if p is None:
                planes[k] = planes[min(got, key=lambda j: abs(j - k))]
    shp = planes[0].shape
    planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]
    vol = np.stack(planes)
    if px and np.isfinite(px) and (px > 0):
        want = int(round(CROP_MM / px))
        h, w = shp
        if 16 < want < min(h, w):
            cy, cx = (h // 2, w // 2)
            half = want // 2
            vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]
    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-06), 0, 1)
    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t = F.interpolate(t, size=(out_size, out_size), mode='bilinear', align_corners=False)
    return (t.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)

In [ ]:
def normalise_laterality(img, plane, lat):
    if lat != 'R':
        return img
    if plane in ('Coronal', 'Axial'):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])

In [ ]:
ORDER_CACHE = os.environ.get('RSNA_ORDER_CACHE') or None

def build_cache(slot_map, plane_map, lat_map, tag):
    studies = sorted(slot_map)
    sidx = {s: i for i, s in enumerate(studies)}
    cache = np.zeros((len(studies), N_SLOT, CACHE_SLICES, IMG, IMG), np.uint8)
    mask = np.zeros((len(studies), N_SLOT), np.float32)
    log(f'{tag}: cache {cache.shape} = {cache.nbytes / 1024 ** 3:.1f} GB')
    jobs = [(st, k, plane, slot_map[st][name]) for st in studies for k, (name, plane, _, _) in enumerate(SLOTS) if name in slot_map[st]]
    n_job = len(jobs)
    t_ord = time.time()
    n_slice_total = sum((len(j[3]['files']) for j in jobs))
    log(f'{tag}: ordering {len(jobs)} slot-series ({n_slice_total} slice headers)')
    ok = done = 0
    CHUNK_O = 1024
    seen = {}
    if ORDER_CACHE and Path(ORDER_CACHE).is_file():
        try:
            import json as _json
            seen = _json.loads(Path(ORDER_CACHE).read_text())
        except (OSError, ValueError):
            seen = {}
        hit = 0
        for _, _, _, rec in jobs:
            e = seen.get(rec['SeriesInstanceUID'])
            if e and len(e['files']) == len(rec['files']):
                rec['ordered'] = e['files']
                ok += int(e['good'])
                hit += 1
        jobs = [j for j in jobs if 'ordered' not in j[3]]
        log(f'{tag}: {hit} slot-series ordered from {ORDER_CACHE}, {len(jobs)} to read')
    with ThreadPoolExecutor(max_workers=ORDER_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK_O):
            block = jobs[c0:c0 + CHUNK_O]
            for (_, _, _, rec), (files, good) in zip(block, pool.map(lambda j: order_slices(j[3]), block)):
                rec['ordered'] = files
                ok += int(good)
                done += 1
                if ORDER_CACHE:
                    seen[rec['SeriesInstanceUID']] = {'files': files, 'good': bool(good)}
            budget = min(ORDER_BUDGET_S, max(60.0, (TIME_BUDGET - (time.time() - T0)) * 0.35))
            if time.time() - t_ord > budget:
                log(f'{tag}: ordering budget spent at {done}/{len(jobs)}; the rest keep file order')
                break
    if ORDER_CACHE and done:
        import json as _json
        _t = Path(ORDER_CACHE).with_suffix('.tmp')
        _t.write_text(_json.dumps(seen))
        _t.replace(Path(ORDER_CACHE))
    log(f'{tag}: ordered {ok}/{n_job} by geometry ({n_job - ok} kept arbitrary) in {time.time() - t_ord:.0f}s')
    jobs = [(st, k, plane, slot_map[st][name]) for st in studies for k, (name, plane, _, _) in enumerate(SLOTS) if name in slot_map[st]]
    log(f'{tag}: decoding {len(jobs)} slot-series')
    n_failed_before = len(DECODE_FAILED)
    CHUNK = 512
    done = 0
    with ThreadPoolExecutor(max_workers=PIX_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK):
            block = jobs[c0:c0 + CHUNK]
            for (st, k, plane, _), img in zip(block, pool.map(lambda j: read_slot(j[3], CACHE_SLICES, IMG), block)):
                done += 1
                if img is None:
                    continue
                cache[sidx[st], k] = normalise_laterality(img, plane, lat_map.get(st)).numpy()
                mask[sidx[st], k] = 1.0
            if done % 4096 < CHUNK:
                log(f'  {tag} {done}/{len(jobs)}')
            if time.time() - T0 > TIME_BUDGET:
                log(f'  {tag}: time budget reached during decode')
                break
    n_failed = len(DECODE_FAILED) - n_failed_before
    log(f'{tag}: {int(mask.sum())}/{len(jobs)} slots filled' + (f'; {n_failed} series had a slice that would not decode' if n_failed else ''))
    gc.collect()
    return (studies, cache, mask)

In [ ]:
# E9: independent RadImageNet ResNet-50 arm for the verified E2 parent.
#
# Adapted 2026-08-11 from the V52 cell in the public Kaggle competition notebook
# prvsiyan/rsna-knee-read-the-report-then-the-knee (latest source SHA-256
# b54aa529f38dc6f594478e7975d86459ddbff898453a2c2633f4c3be4b909e61).
# Kaggle's public-code rule deems public Competition Code open-source; Meta Kaggle
# documents public notebooks under Apache-2.0. Changes here remove the unavailable B3
# arm, pin the public E2 OOF bundle, add per-target/no-regression gates, and preserve E2
# byte-for-byte on every failure. This module is appended as the final notebook cell;
# its imports and DICOM helpers are supplied by the parent notebook.
SLOTS = [
    ("SAG_FS", "Sagittal", None, True),
    ("COR_FS", "Coronal", None, True),
    ("AX_FS", "Axial", None, True),
]
N_SLOT = len(SLOTS)
CACHE_SLICES = 8
TIME_BUDGET = 8.72 * 3600
IMG = CACHE_IMG = 224
# Match the released RadImageNet model's full-frame pretraining. Setting a crop
# larger than every acquisition disables the optional physical crop in read_slot.
CROP_MM = 10_000.0
SLICE_BAND = (0.12, 0.88)
# The audited public OOF was produced after the notebook's legacy member group, which
# left this process-global pixel contract active. Pin it instead of inheriting whichever
# E2 group happened to run last. This uses public preprocessing code only; no legacy
# checkpoint or unknown-license asset is attached.
RULES = dict(RULES_LEGACY)
TOKEN_DIM = 2048             # official ResNet-50 global-average feature
HEAD_DIM = 512
PINNED_HEADS_SHA256 = "0f465649799ecfbccaac1767844639e7ced44e1bc9babde6e4bac7c5d9b89eaa"
PINNED_REMOTE_AUDIT_SHA256 = "267f948078710d3ca8a6f0de4ce0a5e75e850e1f08d36451549a137d878a6fe8"
PINNED_PUBLIC_DIAGNOSTIC_SHA256 = "0f2f82fb40f0570d6766f73b0d7f51489df6d0faa8fd6e4c0f45bcd6c4c7b283"
PINNED_E9B_CONTRACT_SHA256 = "6777c0a0ba7dd044752fac948752dc39e9ca35b3c280fe74ee6d27a5865d87e7"
# E10 repairs E9's censored search: its alpha grid stopped at 0.25 and four of five outer
# folds selected that ceiling, so the deployed 0.20 was a boundary artifact rather than an
# optimum. The ladder, the two-source per-target gains and every deployable weight map live
# in the hash-pinned contract; this cell recomputes the remote half in-kernel before use.
# The contract also carries the held-out form of E10's own weight choice: selecting the rung
# on four grouped folds and scoring the fifth picks 0.60 (public) and 0.70 (v15) in all five
# outer folds, and never picks 0.20. So every deployable rung at or below 0.35 is below what
# honest selection would choose, which is the answer to "you tuned on the 58 gold rows".
PINNED_E10_CONTRACT_SHA256 = "219c91f40905181c222e2966b3fed01a96570ddfb64862357d5fd6cad500cd45"
E10_CONFIG = "uniform_060"
E10_PRESERVED_TARGETS = ["Baker's", "Fracture"]

# E11 trains a third arm whose diversity is in the pixels rather than in the weights. The
# existing arm reads three fat-suppressed slots at full frame; every one of E2's twenty
# members reads one DINOv2 recipe. Nothing in the portfolio has yet looked at a
# non-fat-suppressed series, where meniscal and ligament morphology is conventionally read,
# and nothing has given RadImageNet a physically normalised field of view. E11 changes both:
# three non-suppressed slots plus one suppressed anchor, cropped to 130 mm, which the parent
# notebook establishes is below the acquired field of view of 99.6% of series while still
# containing the joint. It is a training mode only; it never touches the submission.
ARM_MODE = "s0"  # track S: was "e11"
E11_SLOTS = [
    ("SAG_FS", "Sagittal", None, True),
    ("COR_FS", "Coronal", None, True),
    ("AX_FS", "Axial", None, True),
    ("SAG_NOFS", "Sagittal", None, False),
]
E11_CROP_MM = 130.0
E11_CACHE_SLICES = 8
E11_IMG = 224
# Availability of non-suppressed series per plane is unmeasured, so a fill floor rather than
# the parent's 90% rule: below this the run has found something structurally wrong, above it
# an empty slot is simply masked out of the token set like any other absent series.
E11_MIN_FILL = 0.45


def _v52_as_bool(value):
    if pd.isna(value):
        return None
    text = str(value).strip().upper()
    if text in {"1", "TRUE", "T", "YES", "Y"}:
        return True
    if text in {"0", "FALSE", "F", "NO", "N"}:
        return False
    try:
        number = float(text)
        return True if number == 1 else False if number == 0 else None
    except Exception:
        return None


def audit_official_sequence_metadata(inferred, official):
    """Audit metadata agreement without changing the checkpoint pixel contract."""
    needed = {"SeriesInstanceUID", "Fluid_Sensitive", "Fat_Suppression"}
    if inferred.empty or official.empty or not needed.issubset(official.columns):
        return
    inferred_flags = inferred[["SeriesInstanceUID", "fluid", "fatsat"]].copy()
    official_flags = official[
        ["SeriesInstanceUID", "Fluid_Sensitive", "Fat_Suppression"]
    ].copy()
    official_flags["official_fluid"] = official_flags["Fluid_Sensitive"].map(
        _v52_as_bool
    )
    official_flags["official_fatsat"] = official_flags["Fat_Suppression"].map(
        _v52_as_bool
    )
    merged = inferred_flags.merge(
        official_flags[["SeriesInstanceUID", "official_fluid", "official_fatsat"]],
        on="SeriesInstanceUID",
        how="inner",
    )
    for inferred_col, official_col, name in [
        ("fluid", "official_fluid", "Fluid_Sensitive"),
        ("fatsat", "official_fatsat", "Fat_Suppression"),
    ]:
        valid = merged[official_col].notna() & merged[inferred_col].notna()
        if valid.any():
            agreement = (
                merged.loc[valid, inferred_col].astype(bool).to_numpy()
                == merged.loc[valid, official_col].astype(bool).to_numpy()
            ).mean()
            log(
                f"V52 metadata audit {name}: {agreement:.1%} agreement "
                f"on {int(valid.sum())} series"
            )


def find_input_file(name):
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
        if name in files:
            return Path(root) / name
    raise FileNotFoundError(name)


def find_input_dir(name):
    for root, dirs, files in os.walk("/kaggle/input"):
        if Path(root).name == name:
            return Path(root)
    raise FileNotFoundError(name)


def make_targets(train):
    """Three independent public report teachers; image-read gold always wins."""
    uid = "StudyInstanceUID"
    sources = [
        pd.read_csv(find_input_file("report_labels_v2.csv")),
        pd.read_csv(find_input_file("llm_labels_v2.csv")),
        pd.read_csv(find_input_file("labels_llm_gpt56sol.csv")),
    ]
    cube = []
    for frame in sources:
        if frame[uid].duplicated().any():
            raise ValueError("duplicate study in report-label source")
        aligned = train[[uid]].merge(frame[[uid] + TARGETS], on=uid, how="left")
        cube.append(aligned[TARGETS].to_numpy(float))
    cube = np.stack(cube)
    available = np.isfinite(cube).sum(0)
    if np.any(available < 2):
        raise ValueError("fewer than two report teachers for a study/target")
    y = np.nanmean(cube, axis=0).astype(np.float32)
    disagreement = np.nanmean(np.abs(cube - y[None]), axis=0)
    agreement = np.clip(1.0 - 2.0 * disagreement, 0, 1)
    certainty = np.clip(2.0 * np.abs(y - .5), 0, 1)
    w = (.15 + .85 * (.65 * agreement + .35 * certainty)).astype(np.float32)
    gold = train[TARGETS].notna().all(axis=1).to_numpy()
    y[gold] = train.loc[gold, TARGETS].to_numpy(np.float32)
    w[gold] = 3.0
    return y, w, gold


def report_groups(train):
    report = (train.Report.fillna("").astype(str).str.lower()
              .str.replace(r"\s+", " ", regex=True).str.strip())
    return np.array([hashlib.sha256(x.encode()).hexdigest()[:24] for x in report])


def _v52_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(8 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_radimagenet(device):
    """Strictly load the official RadImageNet ResNet-50 PyTorch checkpoint."""
    from torchvision.models import resnet50

    checkpoint = find_input_file("ResNet50.pt")
    expected_checkpoint = "08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734"
    observed_checkpoint = _v52_sha256(checkpoint)
    if observed_checkpoint != expected_checkpoint:
        raise RuntimeError(f"RadImageNet checkpoint drift: {observed_checkpoint}")

    class RadImageNetEncoder(nn.Module):
        def __init__(self):
            super().__init__()
            self.backbone = nn.Sequential(
                *list(resnet50(weights=None).children())[:-2]
            )

        def forward(self, image):
            return self.backbone(image).mean(dim=(2, 3))

    model = RadImageNetEncoder()
    state = torch.load(checkpoint, map_location="cpu", weights_only=True)
    if not state or not all(str(key).startswith("backbone.") for key in state):
        raise RuntimeError("unexpected RadImageNet state-dict namespace")
    model.load_state_dict(state, strict=True)
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    if parameter_count != 23_508_032:
        raise RuntimeError(f"unexpected RadImageNet parameter count {parameter_count}")
    model.eval().to(device)
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    gpu_count = torch.cuda.device_count() if device.type == "cuda" else 0
    if gpu_count > 1:
        model = nn.DataParallel(model, device_ids=list(range(gpu_count)))
    log(
        f"RadImageNet strict load: {parameter_count:,} params; "
        f"inference GPUs={max(1, gpu_count)}"
    )
    return model


@torch.inference_mode()
def encode_radimagenet(cache, slot_mask, device):
    """Encode acquired slices with the official [-1, 1] RadImageNet contract."""
    n, slots, slices, h, w = cache.shape
    features = np.zeros((n, slots * slices, TOKEN_DIM), np.float16)
    token_mask = np.repeat(slot_mask[:, :, None], slices, axis=2).reshape(n, -1)
    valid = np.flatnonzero(token_mask.reshape(-1) > 0)
    flat = cache.reshape(-1, h, w)
    model = load_radimagenet(device)
    if device.type == "cuda":
        batch = 192 if torch.cuda.device_count() > 1 else 96
    else:
        batch = 8
    for b0 in range(0, len(valid), batch):
        ix = valid[b0:b0 + batch]
        x = torch.from_numpy(flat[ix]).to(device).float().div_(127.5).sub_(1.0)
        x = x.unsqueeze(1).expand(-1, 3, -1, -1).contiguous()
        with torch.autocast("cuda", enabled=device.type == "cuda"):
            feat = model(x)
        if feat.shape[1:] != (TOKEN_DIM,):
            raise RuntimeError(f"unexpected RadImageNet feature shape {tuple(feat.shape)}")
        features.reshape(-1, TOKEN_DIM)[ix] = (
            feat.float().cpu().numpy().astype(np.float16)
        )
        if b0 % (batch * 100) == 0:
            log(f"RadImageNet encoded {b0}/{len(valid)} acquired slices")
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return features, token_mask.astype(np.float32)


class FoundationQueryHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.project = nn.Sequential(nn.LayerNorm(TOKEN_DIM),
                                     nn.Linear(TOKEN_DIM, HEAD_DIM), nn.GELU())
        self.plane = nn.Parameter(torch.randn(N_SLOT, HEAD_DIM) * .01)
        self.position = nn.Parameter(torch.randn(CACHE_SLICES, HEAD_DIM) * .01)
        self.query = nn.Parameter(torch.randn(len(TARGETS), HEAD_DIM) * .02)
        self.attn = nn.MultiheadAttention(HEAD_DIM, 8, dropout=.10, batch_first=True)
        self.fuse = nn.Sequential(
            nn.LayerNorm(HEAD_DIM * 4), nn.Linear(HEAD_DIM * 4, HEAD_DIM),
            nn.GELU(), nn.Dropout(.15),
        )
        self.weight = nn.Parameter(torch.randn(len(TARGETS), HEAD_DIM) * .02)
        self.bias = nn.Parameter(torch.zeros(len(TARGETS)))

    def forward(self, feature, mask):
        token = self.project(feature.float())
        token = token.view(len(token), N_SLOT, CACHE_SLICES, HEAD_DIM)
        token = token + self.plane[None, :, None] + self.position[None, None]
        token = token.flatten(1, 2)
        key_padding = mask <= 0
        # No study should be empty, but keep MHA numerically defined if one is.
        all_empty = key_padding.all(1)
        if all_empty.any():
            key_padding = key_padding.clone()
            key_padding[all_empty, 0] = False
        query = self.query.unsqueeze(0).expand(len(token), -1, -1)
        attended = query + self.attn(query, token, token,
                                     key_padding_mask=key_padding,
                                     need_weights=False)[0]
        denom = mask.sum(1, keepdim=True).clamp_min(1).unsqueeze(-1)
        mean = (token * mask.unsqueeze(-1)).sum(1, keepdims=True) / denom
        mean = mean.expand(-1, len(TARGETS), -1)
        fused = self.fuse(torch.cat(
            [attended, mean, torch.abs(attended - mean), attended * mean], -1))
        return (fused * self.weight.unsqueeze(0)).sum(-1) + self.bias


def macro_auc(y, pred):
    from sklearn.metrics import roc_auc_score
    hard = (np.asarray(y) >= .5).astype(np.uint8)
    values = [roc_auc_score(hard[:, j], pred[:, j])
              for j in range(hard.shape[1]) if np.unique(hard[:, j]).size == 2]
    return float(np.mean(values))


def _v52_target_auc(y, pred):
    from sklearn.metrics import roc_auc_score
    hard = (np.asarray(y) >= .5).astype(np.uint8)
    return {
        target: float(roc_auc_score(hard[:, index], pred[:, index]))
        for index, target in enumerate(TARGETS)
    }


@torch.inference_mode()
def predict_head(model, features, masks, indices, device, batch=64):
    model.eval()
    pred = []
    for b0 in range(0, len(indices), batch):
        ix = indices[b0:b0 + batch]
        x = torch.from_numpy(features[ix]).to(device)
        m = torch.from_numpy(masks[ix]).to(device)
        with torch.autocast("cuda", enabled=device.type == "cuda"):
            pred.append(torch.sigmoid(model(x, m)).float().cpu())
    return torch.cat(pred).numpy()


def train_fold(features, masks, y, weights, train_idx, val_idx, fold, device):
    from torch.utils.data import DataLoader, Dataset
    class Rows(Dataset):
        def __init__(self, indices): self.indices = np.asarray(indices)
        def __len__(self): return len(self.indices)
        def __getitem__(self, k):
            i = self.indices[k]
            return features[i], masks[i], y[i], weights[i]
    model = FoundationQueryHead().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=3e-3)
    generator = torch.Generator().manual_seed(SEED + 100 + fold)
    loader = DataLoader(Rows(train_idx), batch_size=48, shuffle=True,
                        generator=generator, num_workers=2, pin_memory=True,
                        persistent_workers=True)
    best, best_auc, stale = None, -1.0, 0
    for epoch in range(24):
        model.train()
        for x, m, target, weight in loader:
            x, m = x.to(device), m.to(device)
            target, weight = target.to(device), weight.to(device)
            with torch.autocast("cuda", enabled=device.type == "cuda"):
                logits = model(x, m)
                raw = F.binary_cross_entropy_with_logits(logits, target,
                                                          reduction="none")
                loss = (raw * weight).sum() / weight.sum().clamp_min(1)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        pred = predict_head(model, features, masks, val_idx, device)
        score = macro_auc(y[val_idx], pred)
        log(f"fold {fold} epoch {epoch}: grouped weak-val AUC {score:.5f}")
        if score > best_auc + 2e-4:
            best_auc, stale = score, 0
            best = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= 5: break
    return best, best_auc


def _v52_rank_columns(values):
    frame = pd.DataFrame(np.asarray(values, dtype=np.float64))
    return frame.rank(method="average", pct=True).to_numpy(np.float64)


def _v52_validate_submission(frame, expected_ids):
    expected_columns = ["StudyInstanceUID", *TARGETS]
    if frame.columns.tolist() != expected_columns:
        raise RuntimeError("V52 submission schema drift")
    ids = frame["StudyInstanceUID"].astype(str).tolist()
    if ids != list(map(str, expected_ids)) or len(ids) != len(set(ids)):
        raise RuntimeError("V52 submission study identity/order drift")
    values = frame[TARGETS].to_numpy(np.float64)
    if not np.isfinite(values).all() or values.min() < 0 or values.max() > 1:
        raise RuntimeError("V52 submission values are invalid")


def main_v52():
    import shutil
    from sklearn.model_selection import GroupKFold

    output = Path("/kaggle/working/rsna_rad_e9")
    output.mkdir(parents=True, exist_ok=True)
    primary = Path("/kaggle/working/submission.csv")
    preserved = Path("/kaggle/working/submission_e2_preserved.csv")
    audit_path = Path("/kaggle/working/rad_e9_audit.json")
    audit = {
        "status": "E2_PRESERVED",
        "evidence_boundary": (
            "All OOF values are local diagnostics on 58 official image labels; "
            "they are not Kaggle competition scores. E2 remains the primary unless "
            "strict artifact, OOF, inference, and submission gates all pass."
        ),
        "encoder": "RadImageNet ResNet-50 official PyTorch release",
        "encoder_license": "CC-BY-NC-SA-4.0 (Kaggle-hosted weight metadata)",
        "encoder_sha256": "08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734",
        "encoder_source_commit": "0ce16f7375db4236e646829d1eca61cdb4282133",
        "base_oof_sha256": "62d47ba4c0c8347b5b24e7fd2aa517aae0fd6d4656fd5d829fd3a50f0159909c",
        "parent": "E2 captured 20-member DINOv2 rank ensemble",
        "blend_contract": "rank columns independently, then 80% E2 plus 20% RadImageNet",
        "pixel_rules": dict(RULES),
    }
    if not primary.is_file():
        raise FileNotFoundError("E2 parent submission is absent")
    shutil.copy2(primary, preserved)

    try:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        if device.type != "cuda":
            raise RuntimeError("V52 RadImageNet experiment requires CUDA")
        elapsed = max(0.0, time.time() - float(globals().get("T0", time.time())))
        available = 8.72 * 3600 - elapsed
        audit["elapsed_before_v52_seconds"] = elapsed
        audit["available_at_start_seconds"] = available
        if available < 2.0 * 3600:
            raise TimeoutError(f"only {available / 60:.1f} minutes remain")

        train = pd.read_csv(ROOT / "train.csv", dtype={"StudyInstanceUID": str})
        train_series = pd.read_csv(
            ROOT / "train_series.csv",
            dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
        )
        if len(train) != 4407:
            raise RuntimeError(f"unexpected train study count {len(train)}")
        plane = dict(zip(train_series.SeriesInstanceUID, train_series.Anatomical_Plane))
        headers = annotate(walk("train_series"))
        audit_official_sequence_metadata(headers, train_series)
        studies, pixels, slot_mask = build_cache(
            pick_slots(headers, plane), plane, lat_of(headers, "train-v52 "), "train-v52"
        )
        by_uid = {str(uid): i for i, uid in enumerate(studies)}
        missing = [uid for uid in train.StudyInstanceUID if uid not in by_uid]
        if missing:
            raise RuntimeError(f"{len(missing)} train studies absent from cache")
        order = np.array([by_uid[uid] for uid in train.StudyInstanceUID], dtype=np.int64)
        pixels, slot_mask = pixels[order], slot_mask[order]
        train_token_count = int(np.repeat(slot_mask[:, :, None], CACHE_SLICES, 2).sum())
        if train_token_count < int(0.90 * len(train) * N_SLOT * CACHE_SLICES):
            raise RuntimeError(f"insufficient acquired train slices: {train_token_count}")
        features, token_mask = encode_radimagenet(pixels, slot_mask, device)
        del pixels, slot_mask, headers
        gc.collect()

        y, weights, gold = make_targets(train)
        if int(gold.sum()) != 58:
            raise RuntimeError(f"expected 58 fully gold studies, observed {int(gold.sum())}")
        groups = report_groups(train)
        if len(np.unique(groups)) < 4000:
            raise RuntimeError("unexpected report-group collapse")

        splits = list(GroupKFold(5).split(features, groups=groups))
        fold_id = np.full(len(train), -1, dtype=np.int8)
        folds = []
        oof = np.zeros_like(y, dtype=np.float32)
        for fold, (tr, va) in enumerate(splits):
            if set(groups[tr]).intersection(groups[va]):
                raise RuntimeError(f"report leakage in fold {fold}")
            fold_id[va] = fold
            state, score = train_fold(
                features, token_mask, y, weights, tr, va, fold, device
            )
            if state is None:
                raise RuntimeError(f"fold {fold} produced no checkpoint")
            head = FoundationQueryHead().to(device)
            head.load_state_dict(state, strict=True)
            oof[va] = predict_head(head, features, token_mask, va, device)
            folds.append({"fold": fold, "weak_auc": float(score), "state_dict": state})
            del head
            torch.cuda.empty_cache()
        if (fold_id < 0).any() or not np.isfinite(oof).all():
            raise RuntimeError("incomplete V52 OOF")

        weak_auc = macro_auc(y, oof)
        gold_auc = macro_auc(y[gold], oof[gold])
        log(f"V52 RadImageNet OOF weak macro AUC {weak_auc:.5f}")
        log(f"V52 RadImageNet OOF gold macro AUC {gold_auc:.5f} on 58 studies")
        torch.save(
            {
                "version": "v52-radimagenet-resnet50-official-1",
                "targets": TARGETS,
                "encoder_sha256": audit["encoder_sha256"],
                "encoder_source_commit": audit["encoder_source_commit"],
                "img": IMG,
                "slices_per_plane": CACHE_SLICES,
                "feature": "global_average_pool",
                "folds": folds,
                "weak_oof_auc": weak_auc,
                "gold_oof_auc": gold_auc,
            },
            output / "v52_radimagenet_heads.pt",
        )
        oof_frame = pd.DataFrame(oof, columns=TARGETS)
        oof_frame.insert(0, "StudyInstanceUID", train.StudyInstanceUID)
        oof_frame["fold"] = fold_id
        oof_frame["is_gold"] = gold.astype(np.uint8)
        oof_frame.to_csv(output / "v52_oof.csv", index=False)

        base_npz = find_input_file("oof.npz")
        observed_base_hash = _v52_sha256(base_npz)
        if observed_base_hash != audit["base_oof_sha256"]:
            raise RuntimeError(f"E2 OOF artifact drift: {observed_base_hash}")
        with np.load(base_npz, allow_pickle=False) as base_bundle:
            expected_members = {"ids", "pred", "y_derived", "gold_mask", "targets"}
            if set(base_bundle.files) != expected_members:
                raise RuntimeError(f"unexpected E2 OOF members: {base_bundle.files}")
            base_ids = base_bundle["ids"].astype(str)
            base_targets = base_bundle["targets"].astype(str).tolist()
            base_gold = base_bundle["gold_mask"].astype(bool)
            base_prediction = base_bundle["pred"].astype(np.float64)
        if base_targets != TARGETS:
            raise RuntimeError("E2 OOF target order drift")
        if not np.array_equal(base_ids, train.StudyInstanceUID.astype(str).to_numpy()):
            raise RuntimeError("E2 OOF study order drift")
        if not np.array_equal(base_gold, gold):
            raise RuntimeError("E2 OOF gold mask differs from official train.csv")
        train_rows = np.flatnonzero(gold)
        if len(train_rows) != 58:
            raise RuntimeError(f"expected 58 E2 gold rows, observed {len(train_rows)}")
        gold_y = train.loc[gold, TARGETS].to_numpy(np.float64)
        exact_public = base_prediction[gold]
        rad = oof[gold].astype(np.float64)
        if not all(np.isfinite(x).all() for x in (gold_y, exact_public, rad)):
            raise RuntimeError("non-finite aligned E2/RadImageNet OOF value")

        base_rank = _v52_rank_columns(exact_public)
        rad_rank = _v52_rank_columns(rad)
        base_score = macro_auc(gold_y, base_rank)
        rad_score = macro_auc(gold_y, rad_rank)
        alpha_grid = np.array([0.0, 0.025, 0.05, 0.10, 0.15, 0.20, 0.25])
        gold_folds = fold_id[train_rows]
        if sorted(np.unique(gold_folds).tolist()) != [0, 1, 2, 3, 4]:
            raise RuntimeError("gold rows do not cover all five grouped folds")
        nested = np.zeros_like(base_rank)
        choices = []
        outer_train_scores = []
        for outer in range(5):
            tr = gold_folds != outer
            va = ~tr
            scored = []
            for alpha in alpha_grid:
                blend = (1.0 - alpha) * base_rank[tr] + alpha * rad_rank[tr]
                score = macro_auc(gold_y[tr], blend) - 0.01 * float(alpha)
                scored.append(float(score))
            best = max(range(len(alpha_grid)), key=lambda i: (scored[i], -alpha_grid[i]))
            alpha = float(alpha_grid[best])
            choices.append(alpha)
            outer_train_scores.append(scored)
            nested[va] = (1.0 - alpha) * base_rank[va] + alpha * rad_rank[va]
        nested_score = macro_auc(gold_y, nested)
        # Deployment weight is fixed by the independently scored public 0.906 mechanism.
        # The 58 gold rows may veto it and measure fold stability, but do not tune it.
        final_alpha = 0.20
        final_oof = (1.0 - final_alpha) * base_rank + final_alpha * rad_rank
        final_score = macro_auc(gold_y, final_oof)
        grid_scores = {
            f"{alpha:.3f}": macro_auc(
                gold_y, (1.0 - alpha) * base_rank + alpha * rad_rank
            )
            for alpha in alpha_grid
        }
        base_target_scores = _v52_target_auc(gold_y, base_rank)
        rad_target_scores = _v52_target_auc(gold_y, rad_rank)
        final_target_scores = _v52_target_auc(gold_y, final_oof)
        target_deltas = {
            target: final_target_scores[target] - base_target_scores[target]
            for target in TARGETS
        }
        target_regressions = {
            target: delta for target, delta in target_deltas.items() if delta < -1e-12
        }
        positive_folds = int(sum(alpha > 0 for alpha in choices))
        supported = bool(
            final_alpha > 0
            and positive_folds >= 3
            and nested_score >= base_score + 0.001
            and final_score >= base_score + 0.001
            and not target_regressions
        )
        audit["oof"] = {
            "rows": 58,
            "weak_macro_auc": weak_auc,
            "rad_gold_macro_auc": rad_score,
            "e2_macro_auc": base_score,
            "outer_fold_choices": choices,
            "outer_fold_penalized_train_scores": outer_train_scores,
            "nested_blend_macro_auc": nested_score,
            "final_alpha": final_alpha,
            "final_descriptive_macro_auc": final_score,
            "full_grid_macro_auc": grid_scores,
            "positive_outer_folds": positive_folds,
            "per_target": {
                target: {
                    "e2_auc": base_target_scores[target],
                    "radimagenet_auc": rad_target_scores[target],
                    "blend_auc": final_target_scores[target],
                    "blend_delta": target_deltas[target],
                }
                for target in TARGETS
            },
            "target_regressions": target_regressions,
            "gold_fold_counts": {
                str(fold): int((gold_folds == fold).sum()) for fold in range(5)
            },
            "selection_supported": supported,
        }
        audit["train_available_slice_tokens"] = train_token_count
        audit["head_count"] = len(folds)
        if not supported:
            audit["status"] = "OOF_REJECTED_E2_PRESERVED"
            log(
                f"V52 rejected by nested OOF: base={base_score:.5f}, "
                f"nested={nested_score:.5f}, final={final_score:.5f}, choices={choices}"
            )
            return

        del features, token_mask
        gc.collect()
        test = pd.read_csv(ROOT / "test.csv", dtype={"StudyInstanceUID": str})
        test_series = pd.read_csv(
            ROOT / "test_series.csv",
            dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
        )
        test_plane = dict(zip(test_series.SeriesInstanceUID, test_series.Anatomical_Plane))
        test_headers = annotate(walk("test_series"))
        audit_official_sequence_metadata(test_headers, test_series)
        test_studies, test_pixels, test_slot_mask = build_cache(
            pick_slots(test_headers, test_plane),
            test_plane,
            lat_of(test_headers, "test-v52 "),
            "test-v52",
        )
        test_by_uid = {str(uid): i for i, uid in enumerate(test_studies)}
        test_missing = [uid for uid in test.StudyInstanceUID if uid not in test_by_uid]
        if test_missing:
            raise RuntimeError(f"{len(test_missing)} test studies absent from cache")
        test_order = np.array([test_by_uid[uid] for uid in test.StudyInstanceUID])
        test_pixels = test_pixels[test_order]
        test_slot_mask = test_slot_mask[test_order]
        test_token_count = int(
            np.repeat(test_slot_mask[:, :, None], CACHE_SLICES, 2).sum()
        )
        if test_token_count < int(0.85 * len(test) * N_SLOT * CACHE_SLICES):
            raise RuntimeError(f"insufficient acquired test slices: {test_token_count}")
        test_features, test_token_mask = encode_radimagenet(
            test_pixels, test_slot_mask, device
        )
        del test_pixels, test_slot_mask, test_headers
        gc.collect()

        fold_predictions = []
        all_test = np.arange(len(test), dtype=np.int64)
        for record in folds:
            head = FoundationQueryHead().to(device)
            head.load_state_dict(record["state_dict"], strict=True)
            fold_predictions.append(
                predict_head(head, test_features, test_token_mask, all_test, device)
            )
            del head
            torch.cuda.empty_cache()
        if len(fold_predictions) != 5:
            raise RuntimeError("test inference did not use all five heads")
        rad_test = np.mean(np.stack(fold_predictions), axis=0)
        if not np.isfinite(rad_test).all():
            raise RuntimeError("non-finite RadImageNet test prediction")

        baseline = pd.read_csv(preserved, dtype={"StudyInstanceUID": str})
        _v52_validate_submission(baseline, test.StudyInstanceUID)
        rad_frame = pd.DataFrame(rad_test, columns=TARGETS)
        rad_frame.insert(0, "StudyInstanceUID", test.StudyInstanceUID)
        _v52_validate_submission(rad_frame, test.StudyInstanceUID)
        rad_frame.to_csv(output / "submission_rad_only.csv", index=False)
        baseline_rank = _v52_rank_columns(baseline[TARGETS].to_numpy())
        rad_test_rank = _v52_rank_columns(rad_test)
        selected_path = None
        for alpha in alpha_grid[1:]:
            candidate = baseline.copy()
            candidate[TARGETS] = (
                (1.0 - alpha) * baseline_rank + alpha * rad_test_rank
            )
            _v52_validate_submission(candidate, test.StudyInstanceUID)
            path = output / f"submission_e2_rad_{int(round(1000 * alpha)):03d}.csv"
            candidate.to_csv(path, index=False)
            if abs(float(alpha) - final_alpha) < 1e-12:
                selected_path = path
        if selected_path is None or not selected_path.is_file():
            raise RuntimeError("selected V52 blend artifact is absent")
        selected = pd.read_csv(selected_path, dtype={"StudyInstanceUID": str})
        _v52_validate_submission(selected, test.StudyInstanceUID)
        audit["test_studies"] = len(test)
        audit["test_available_slice_tokens"] = test_token_count
        audit["test_head_count"] = len(fold_predictions)
        audit["selected_path"] = str(selected_path)
        audit["selected_sha256"] = _v52_sha256(selected_path)
        audit["fallback_sha256"] = _v52_sha256(preserved)
        shutil.copy2(selected_path, primary)
        if _v52_sha256(primary) != audit["selected_sha256"]:
            raise RuntimeError("primary V52 copy hash mismatch")
        audit["status"] = "CANDIDATE_SELECTED"
        log(
            f"E9 selected alpha={final_alpha:.3f}; "
            f"nested={nested_score:.5f} vs E2 OOF={base_score:.5f}"
        )
    except Exception as error:
        audit["status"] = "ERROR_E2_PRESERVED"
        audit["error"] = f"{type(error).__name__}: {error}"
        audit["traceback"] = traceback.format_exc()
        log(f"E9 preserves E2: {audit['error']}")
    finally:
        if audit.get("status") != "CANDIDATE_SELECTED" and preserved.is_file():
            shutil.copy2(preserved, primary)
        audit["primary_sha256"] = _v52_sha256(primary) if primary.is_file() else None
        audit_path.write_text(json.dumps(audit, indent=2, sort_keys=True) + "\n")


def _v52_load_pinned_e9b():
    """Load the public v15 heads and reconstruct the dual-OOF target gate."""
    heads_path = find_input_file("v52_radimagenet_heads.pt")
    remote_path = find_input_file("rad_e9_audit.json")
    public_path = find_input_file("public_oof_diagnostic.json")
    contract_path = find_input_file("e9b_contract.json")
    expected_hashes = {
        heads_path: PINNED_HEADS_SHA256,
        remote_path: PINNED_REMOTE_AUDIT_SHA256,
        public_path: PINNED_PUBLIC_DIAGNOSTIC_SHA256,
        contract_path: PINNED_E9B_CONTRACT_SHA256,
    }
    for path, expected in expected_hashes.items():
        observed = _v52_sha256(path)
        if observed != expected:
            raise RuntimeError(f"pinned E9b artifact drift for {path.name}: {observed}")

    remote = json.loads(remote_path.read_text())
    public = json.loads(public_path.read_text())
    contract = json.loads(contract_path.read_text())
    if remote.get("status") != "OOF_REJECTED_E2_PRESERVED":
        raise RuntimeError(f"unexpected v15 audit status {remote.get('status')}")
    if remote.get("primary_sha256") != (
        "f9fb57b7bac8489a5d5285b3984b06df57f142572be6417eac6341c43e96707a"
    ):
        raise RuntimeError("v15 did not preserve the exact E2 visible artifact")
    remote_oof = remote.get("oof", {})
    public_target = public.get("per_target", {})
    remote_target = remote_oof.get("per_target", {})
    if set(public_target) != set(TARGETS) or set(remote_target) != set(TARGETS):
        raise RuntimeError("E9b diagnostic target set drift")
    if int(remote.get("head_count", -1)) != 5:
        raise RuntimeError("v15 remote audit does not contain five heads")
    if int(remote_oof.get("positive_outer_folds", -1)) != 5:
        raise RuntimeError("v15 remote outer-fold support drift")
    if int(public.get("positive_outer_folds", -1)) != 5:
        raise RuntimeError("public outer-fold support drift")

    selected_targets = [
        target for target in TARGETS
        if float(public_target[target]["blend_delta"]) > 1e-12
        and float(remote_target[target]["blend_delta"]) > 1e-12
    ]
    preserved_targets = [target for target in TARGETS if target not in selected_targets]
    if selected_targets != contract.get("selected_targets"):
        raise RuntimeError(f"E9b selected-target contract drift: {selected_targets}")
    if preserved_targets != contract.get("preserved_targets"):
        raise RuntimeError(f"E9b preserved-target contract drift: {preserved_targets}")
    if len(selected_targets) != 10 or preserved_targets != ["Baker's", "Fracture"]:
        raise RuntimeError("E9b requires the ten-target dual-OOF intersection")
    alpha = float(contract.get("alpha", -1))
    if abs(alpha - 0.20) > 1e-12:
        raise RuntimeError(f"E9b alpha drift: {alpha}")

    def selective_macro(records, base_key, blend_key):
        return float(np.mean([
            float(records[target][blend_key] if target in selected_targets
                  else records[target][base_key])
            for target in TARGETS
        ]))

    public_base = float(public["base_gold_macro_auc"])
    remote_base = float(remote_oof["e2_macro_auc"])
    public_selective = selective_macro(public_target, "base_auc", "blend_auc")
    remote_selective = selective_macro(remote_target, "e2_auc", "blend_auc")
    if public_selective < public_base + 0.001:
        raise RuntimeError("E9b public selective gate no longer improves E2")
    if remote_selective < remote_base + 0.001:
        raise RuntimeError("E9b remote selective gate no longer improves E2")

    payload = torch.load(heads_path, map_location="cpu", weights_only=True)
    expected_payload = {
        "version": "v52-radimagenet-resnet50-official-1",
        "targets": TARGETS,
        "encoder_sha256": (
            "08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734"
        ),
        "encoder_source_commit": "0ce16f7375db4236e646829d1eca61cdb4282133",
        "img": 224,
        "slices_per_plane": 8,
        "feature": "global_average_pool",
    }
    for key, expected in expected_payload.items():
        if payload.get(key) != expected:
            raise RuntimeError(f"pinned E9b head contract drift for {key}")
    folds = payload.get("folds")
    if not isinstance(folds, list) or len(folds) != 5:
        raise RuntimeError("pinned E9b payload requires five folds")
    if sorted(int(record.get("fold", -1)) for record in folds) != list(range(5)):
        raise RuntimeError("pinned E9b fold identity drift")
    if any(not isinstance(record.get("state_dict"), dict) for record in folds):
        raise RuntimeError("pinned E9b state dictionary is absent")
    if abs(float(payload.get("weak_oof_auc", -1)) - 0.8278261335825697) > 1e-12:
        raise RuntimeError("pinned E9b weak OOF drift")
    if abs(float(payload.get("gold_oof_auc", -1)) - 0.8543239133509962) > 1e-12:
        raise RuntimeError("pinned E9b gold OOF drift")
    return {
        "payload": payload,
        "folds": folds,
        "alpha": alpha,
        "selected_targets": selected_targets,
        "preserved_targets": preserved_targets,
        "public_base": public_base,
        "public_selective": public_selective,
        "remote_base": remote_base,
        "remote_selective": remote_selective,
        "remote_outer_fold_choices": remote_oof["outer_fold_choices"],
    }


def _v52_e10_remote_ladder(contract):
    """Recompute this account's half of the ladder from artifacts the kernel can read.

    The contract carries per-target gains for two independent RadImageNet OOF runs. Only the
    public run is unverifiable here, so its numbers stay data. The remote run is rebuilt from
    the attached OOF table, the pinned E2 OOF bundle and the official labels, and must match
    the contract exactly or E10 refuses to deploy.
    """
    train = pd.read_csv(ROOT / "train.csv", dtype={"StudyInstanceUID": str})
    gold = train[TARGETS].notna().all(axis=1).to_numpy()
    oof_path = find_input_file("v52_oof.csv")
    observed = _v52_sha256(oof_path)
    if observed != contract["remote_oof_sha256"]:
        raise RuntimeError(f"E10 remote OOF drift: {observed}")
    rad_frame = pd.read_csv(oof_path, dtype={"StudyInstanceUID": str})
    if rad_frame.columns.tolist() != ["StudyInstanceUID", *TARGETS, "fold", "is_gold"]:
        raise RuntimeError("E10 remote OOF schema drift")
    aligned = train[["StudyInstanceUID"]].merge(
        rad_frame, on="StudyInstanceUID", how="left", validate="one_to_one"
    )
    if aligned[TARGETS].isna().any().any():
        raise RuntimeError("E10 remote OOF does not cover every official train study")

    base_npz = find_input_file("oof.npz")
    with np.load(base_npz, allow_pickle=False) as bundle:
        if bundle["targets"].astype(str).tolist() != TARGETS:
            raise RuntimeError("E10 E2 OOF target order drift")
        if not np.array_equal(
            bundle["ids"].astype(str), train.StudyInstanceUID.astype(str).to_numpy()
        ):
            raise RuntimeError("E10 E2 OOF study order drift")
        if not np.array_equal(bundle["gold_mask"].astype(bool), gold):
            raise RuntimeError("E10 E2 gold mask differs from official train.csv")
        base_prediction = bundle["pred"].astype(np.float64)

    # Rank within the scored rows, matching both the E9b parent and test-time deployment
    # where the ranked population and the scored population are the same studies.
    base = _v52_rank_columns(base_prediction[gold])
    rad = _v52_rank_columns(aligned[TARGETS].to_numpy(np.float64)[gold])
    gold_y = train.loc[gold, TARGETS].to_numpy(np.float64)
    if len(gold_y) != 58 or not np.isfinite(base).all() or not np.isfinite(rad).all():
        raise RuntimeError("E10 gold alignment is incomplete or non-finite")
    reference = _v52_target_auc(gold_y, base)
    if abs(
        float(np.mean([reference[t] for t in TARGETS]))
        - float(contract["base_gold_macro_auc"])
    ) > 1e-9:
        raise RuntimeError("E10 base gold diagnostic drift")

    rebuilt = {}
    for key in contract["ladder"]:
        alpha = float(key)
        scores = _v52_target_auc(gold_y, (1.0 - alpha) * base + alpha * rad)
        rebuilt[key] = {t: scores[t] - reference[t] for t in TARGETS}
    pinned_remote = contract["per_target_ladder_delta"]["remote_v15"]
    if set(rebuilt) != set(pinned_remote):
        raise RuntimeError("E10 ladder key drift")
    for key, deltas in rebuilt.items():
        for target, delta in deltas.items():
            if abs(delta - float(pinned_remote[key][target])) > 1e-9:
                raise RuntimeError(
                    f"E10 recomputed remote gain disagrees at {key}/{target}: {delta}"
                )
    if any(abs(reference[t] - float(contract["per_target_base_auc"][t])) > 1e-9 for t in TARGETS):
        raise RuntimeError("E10 per-target base AUC drift")
    return rebuilt, reference


def _v52_load_e10():
    """Validate the E10 contract, then return the weight map the kernel will deploy."""
    heads_path = find_input_file("v52_radimagenet_heads.pt")
    remote_path = find_input_file("rad_e9_audit.json")
    contract_path = find_input_file("e10_contract.json")
    for path, expected in (
        (heads_path, PINNED_HEADS_SHA256),
        (remote_path, PINNED_REMOTE_AUDIT_SHA256),
        (contract_path, PINNED_E10_CONTRACT_SHA256),
    ):
        observed = _v52_sha256(path)
        if observed != expected:
            raise RuntimeError(f"pinned E10 artifact drift for {path.name}: {observed}")

    contract = json.loads(contract_path.read_text())
    if contract.get("version") != "e10-alpha-ladder-2":
        raise RuntimeError(f"unexpected E10 contract version {contract.get('version')}")
    if contract.get("targets") != TARGETS:
        raise RuntimeError("E10 contract target order drift")
    remote = json.loads(remote_path.read_text())
    if remote.get("status") != "OOF_REJECTED_E2_PRESERVED":
        raise RuntimeError(f"unexpected v15 audit status {remote.get('status')}")
    if remote.get("primary_sha256") != (
        "f9fb57b7bac8489a5d5285b3984b06df57f142572be6417eac6341c43e96707a"
    ):
        raise RuntimeError("v15 did not preserve the exact E2 visible artifact")
    if int(remote.get("head_count", -1)) != 5:
        raise RuntimeError("v15 remote audit does not contain five heads")

    rebuilt, base_auc = _v52_e10_remote_ladder(contract)
    public_ladder = contract["per_target_ladder_delta"]["public"]
    configuration = contract["configurations"].get(E10_CONFIG)
    if configuration is None:
        raise RuntimeError(f"E10 contract has no configuration {E10_CONFIG!r}")
    alpha_map = {t: float(configuration["alpha_map"][t]) for t in TARGETS}
    if any(alpha < 0.0 or alpha > 1.0 for alpha in alpha_map.values()):
        raise RuntimeError("E10 weight outside the unit interval")
    preserved = sorted(t for t, alpha in alpha_map.items() if alpha == 0.0)
    if preserved != sorted(E10_PRESERVED_TARGETS):
        raise RuntimeError(f"E10 preserved-target drift: {preserved}")

    # Two-tier gate. The scored objective is macro AUC, so the binding requirement is that
    # the deployed map raise the macro in BOTH independent runs -- the public numbers as
    # pinned data, this account's numbers as recomputed above. A per-target "never harm any
    # single label" rule is strictly stronger than that objective and would veto rungs that
    # trade a small loss on one label for a large gain on another, so it is enforced only for
    # configurations that actually claim it. Whichever claim the contract makes is verified;
    # a configuration cannot quietly assert dual-positivity it no longer has.
    claims_dual_positive = bool(configuration["all_dual_positive"])
    observed_dual_positive = True
    for target, alpha in alpha_map.items():
        if alpha == 0.0:
            continue
        key = f"{alpha:.2f}"
        if key not in rebuilt:
            raise RuntimeError(f"E10 weight {key} is outside the audited ladder")
        gains = (float(public_ladder[key][target]), float(rebuilt[key][target]))
        if not all(gain > 0 for gain in gains):
            observed_dual_positive = False
            if claims_dual_positive:
                raise RuntimeError(
                    f"E10 dual-source gate rejects {target} at {key}: {gains}"
                )
    if observed_dual_positive != claims_dual_positive:
        raise RuntimeError(
            f"E10 contract claims all_dual_positive={claims_dual_positive} for "
            f"{E10_CONFIG!r} but recomputation observes {observed_dual_positive}"
        )

    macro = {}
    for name, ladder in (("public", public_ladder), ("remote_v15", rebuilt)):
        total = 0.0
        for target, alpha in alpha_map.items():
            gain = 0.0 if alpha == 0.0 else float(ladder[f"{alpha:.2f}"][target])
            total += float(base_auc[target]) + gain
        macro[name] = total / len(TARGETS)
        if macro[name] <= float(contract["base_gold_macro_auc"]):
            raise RuntimeError(
                f"E10 macro gate rejects {E10_CONFIG!r}: {name} macro {macro[name]} "
                f"does not beat base {contract['base_gold_macro_auc']}"
            )
        if abs(macro[name] - float(configuration["descriptive_macro"][name])) > 1e-9:
            raise RuntimeError(
                f"E10 recomputed {name} macro {macro[name]} disagrees with the contract "
                f"value {configuration['descriptive_macro'][name]}"
            )

    payload = torch.load(heads_path, map_location="cpu", weights_only=True)
    expected_payload = {
        "version": "v52-radimagenet-resnet50-official-1",
        "targets": TARGETS,
        "encoder_sha256": (
            "08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734"
        ),
        "encoder_source_commit": "0ce16f7375db4236e646829d1eca61cdb4282133",
        "img": 224,
        "slices_per_plane": 8,
        "feature": "global_average_pool",
    }
    for key, expected in expected_payload.items():
        if payload.get(key) != expected:
            raise RuntimeError(f"pinned E10 head contract drift for {key}")
    folds = payload.get("folds")
    if not isinstance(folds, list) or len(folds) != 5:
        raise RuntimeError("pinned E10 payload requires five folds")
    if sorted(int(record.get("fold", -1)) for record in folds) != list(range(5)):
        raise RuntimeError("pinned E10 fold identity drift")
    if any(not isinstance(record.get("state_dict"), dict) for record in folds):
        raise RuntimeError("pinned E10 state dictionary is absent")
    if abs(float(payload.get("gold_oof_auc", -1)) - 0.8543239133509962) > 1e-12:
        raise RuntimeError("pinned E10 gold OOF drift")
    return {
        "payload": payload,
        "folds": folds,
        "contract": contract,
        "configuration": E10_CONFIG,
        "alpha_map": alpha_map,
        "preserved_targets": sorted(E10_PRESERVED_TARGETS),
        "diagnostic_macro": configuration["diagnostic_macro"],
        "recomputed_macro": macro,
        "all_dual_positive": claims_dual_positive,
        "rationale": configuration["rationale"],
        "recomputed_remote_ladder": rebuilt,
    }


def main_v52_pinned_e9b():
    """Inference-only E9b from hash-pinned v15 heads; preserve E2 on any failure."""
    import shutil

    output = Path("/kaggle/working/rsna_rad_e9b")
    output.mkdir(parents=True, exist_ok=True)
    primary = Path("/kaggle/working/submission.csv")
    preserved = Path("/kaggle/working/submission_e2_preserved.csv")
    audit_path = Path("/kaggle/working/rad_e9b_audit.json")
    audit = {
        "status": "E2_PRESERVED",
        "mode": "pinned_v15_heads_inference_only",
        "evidence_boundary": (
            "OOF values are diagnostics, not Kaggle scores. The fixed public 20-percent "
            "vote is applied only to targets improving in two independent OOF runs."
        ),
        "encoder": "RadImageNet ResNet-50 official PyTorch release",
        "encoder_license": "CC-BY-NC-SA-4.0",
        "encoder_sha256": (
            "08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734"
        ),
        "heads_sha256": PINNED_HEADS_SHA256,
        "parent": "E2 captured 20-member DINOv2 rank ensemble",
        "pixel_rules": dict(RULES),
    }
    if not primary.is_file():
        raise FileNotFoundError("E2 parent submission is absent")
    shutil.copy2(primary, preserved)

    try:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        if device.type != "cuda":
            raise RuntimeError("E9b RadImageNet inference requires CUDA")
        elapsed = max(0.0, time.time() - float(globals().get("T0", time.time())))
        available = 8.72 * 3600 - elapsed
        audit["elapsed_before_e9b_seconds"] = elapsed
        audit["available_at_start_seconds"] = available
        if available < 45 * 60:
            raise TimeoutError(f"only {available / 60:.1f} minutes remain")

        pinned = _v52_load_pinned_e9b()
        audit["oof_gate"] = {
            "alpha": pinned["alpha"],
            "selected_targets": pinned["selected_targets"],
            "preserved_targets": pinned["preserved_targets"],
            "public_e2_macro_auc": pinned["public_base"],
            "public_selective_macro_auc": pinned["public_selective"],
            "remote_e2_macro_auc": pinned["remote_base"],
            "remote_selective_macro_auc": pinned["remote_selective"],
            "remote_outer_fold_choices": pinned["remote_outer_fold_choices"],
            "selection_supported": True,
        }

        test = pd.read_csv(ROOT / "test.csv", dtype={"StudyInstanceUID": str})
        test_series = pd.read_csv(
            ROOT / "test_series.csv",
            dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
        )
        plane = dict(zip(test_series.SeriesInstanceUID, test_series.Anatomical_Plane))
        headers = annotate(walk("test_series"))
        audit_official_sequence_metadata(headers, test_series)
        studies, pixels, slot_mask = build_cache(
            pick_slots(headers, plane), plane, lat_of(headers, "test-e9b "), "test-e9b"
        )
        by_uid = {str(uid): index for index, uid in enumerate(studies)}
        missing = [uid for uid in test.StudyInstanceUID if uid not in by_uid]
        if missing:
            raise RuntimeError(f"{len(missing)} test studies absent from E9b cache")
        order = np.asarray([by_uid[uid] for uid in test.StudyInstanceUID], dtype=np.int64)
        pixels, slot_mask = pixels[order], slot_mask[order]
        token_count = int(np.repeat(slot_mask[:, :, None], CACHE_SLICES, 2).sum())
        if token_count < int(0.85 * len(test) * N_SLOT * CACHE_SLICES):
            raise RuntimeError(f"insufficient acquired E9b test slices: {token_count}")
        features, token_mask = encode_radimagenet(pixels, slot_mask, device)
        del pixels, slot_mask, headers
        gc.collect()

        all_test = np.arange(len(test), dtype=np.int64)
        fold_predictions = []
        for record in pinned["folds"]:
            head = FoundationQueryHead().to(device)
            head.load_state_dict(record["state_dict"], strict=True)
            fold_predictions.append(
                predict_head(head, features, token_mask, all_test, device)
            )
            del head
            torch.cuda.empty_cache()
        if len(fold_predictions) != 5:
            raise RuntimeError("E9b test inference did not use all five heads")
        rad_test = np.mean(np.stack(fold_predictions), axis=0)
        if rad_test.shape != (len(test), len(TARGETS)) or not np.isfinite(rad_test).all():
            raise RuntimeError(f"invalid E9b prediction shape/value: {rad_test.shape}")

        baseline = pd.read_csv(preserved, dtype={"StudyInstanceUID": str})
        _v52_validate_submission(baseline, test.StudyInstanceUID)
        rad_frame = pd.DataFrame(rad_test, columns=TARGETS)
        rad_frame.insert(0, "StudyInstanceUID", test.StudyInstanceUID)
        _v52_validate_submission(rad_frame, test.StudyInstanceUID)
        rad_frame.to_csv(output / "submission_rad_only.csv", index=False)
        baseline_rank = _v52_rank_columns(baseline[TARGETS].to_numpy())
        rad_rank = _v52_rank_columns(rad_test)
        selected = baseline.copy()
        alpha = pinned["alpha"]
        for target in pinned["selected_targets"]:
            index = TARGETS.index(target)
            selected[target] = (
                (1.0 - alpha) * baseline_rank[:, index] + alpha * rad_rank[:, index]
            )
        for target in pinned["preserved_targets"]:
            if not np.array_equal(
                selected[target].to_numpy(), baseline[target].to_numpy()
            ):
                raise RuntimeError(f"E9b failed to preserve {target}")
        _v52_validate_submission(selected, test.StudyInstanceUID)
        selected_path = output / "submission_e2_rad_robust_200.csv"
        selected.to_csv(selected_path, index=False)
        selected = pd.read_csv(selected_path, dtype={"StudyInstanceUID": str})
        _v52_validate_submission(selected, test.StudyInstanceUID)

        audit.update({
            "test_studies": len(test),
            "test_available_slice_tokens": token_count,
            "test_head_count": len(fold_predictions),
            "selected_path": str(selected_path),
            "selected_sha256": _v52_sha256(selected_path),
            "fallback_sha256": _v52_sha256(preserved),
        })
        shutil.copy2(selected_path, primary)
        if _v52_sha256(primary) != audit["selected_sha256"]:
            raise RuntimeError("primary E9b copy hash mismatch")
        audit["status"] = "CANDIDATE_SELECTED"
        log(
            f"E9b selected alpha={alpha:.3f} on "
            f"{len(pinned['selected_targets'])} dual-OOF-stable targets; "
            "Baker's and Fracture preserve E2"
        )
    except Exception as error:
        audit["status"] = "ERROR_E2_PRESERVED"
        audit["error"] = f"{type(error).__name__}: {error}"
        audit["traceback"] = traceback.format_exc()
        log(f"E9b preserves E2: {audit['error']}")
    finally:
        if audit.get("status") != "CANDIDATE_SELECTED" and preserved.is_file():
            shutil.copy2(preserved, primary)
        audit["primary_sha256"] = _v52_sha256(primary) if primary.is_file() else None
        audit_path.write_text(json.dumps(audit, indent=2, sort_keys=True) + "\n")


def _v52_rad_test_predictions(pinned, test, device, tag):
    """Five-head RadImageNet test prediction on the official test tree."""
    test_series = pd.read_csv(
        ROOT / "test_series.csv",
        dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
    )
    plane = dict(zip(test_series.SeriesInstanceUID, test_series.Anatomical_Plane))
    headers = annotate(walk("test_series"))
    audit_official_sequence_metadata(headers, test_series)
    studies, pixels, slot_mask = build_cache(
        pick_slots(headers, plane), plane, lat_of(headers, f"{tag} "), tag
    )
    by_uid = {str(uid): index for index, uid in enumerate(studies)}
    missing = [uid for uid in test.StudyInstanceUID if uid not in by_uid]
    if missing:
        raise RuntimeError(f"{len(missing)} test studies absent from {tag} cache")
    order = np.asarray([by_uid[uid] for uid in test.StudyInstanceUID], dtype=np.int64)
    pixels, slot_mask = pixels[order], slot_mask[order]
    token_count = int(np.repeat(slot_mask[:, :, None], CACHE_SLICES, 2).sum())
    if token_count < int(0.85 * len(test) * N_SLOT * CACHE_SLICES):
        raise RuntimeError(f"insufficient acquired {tag} test slices: {token_count}")
    features, token_mask = encode_radimagenet(pixels, slot_mask, device)
    del pixels, slot_mask, headers
    gc.collect()

    rows = np.arange(len(test), dtype=np.int64)
    predictions = []
    for record in pinned["folds"]:
        head = FoundationQueryHead().to(device)
        head.load_state_dict(record["state_dict"], strict=True)
        predictions.append(predict_head(head, features, token_mask, rows, device))
        del head
        torch.cuda.empty_cache()
    if len(predictions) != 5:
        raise RuntimeError(f"{tag} test inference did not use all five heads")
    rad_test = np.mean(np.stack(predictions), axis=0)
    if rad_test.shape != (len(test), len(TARGETS)) or not np.isfinite(rad_test).all():
        raise RuntimeError(f"invalid {tag} prediction shape/value: {rad_test.shape}")
    return rad_test, token_count, len(predictions)


def main_v52_e10():
    """Deploy the audited E10 weight map; preserve the E2 parent on any failure."""
    import shutil

    output = Path("/kaggle/working/rsna_rad_e10")
    output.mkdir(parents=True, exist_ok=True)
    primary = Path("/kaggle/working/submission.csv")
    preserved = Path("/kaggle/working/submission_e2_preserved.csv")
    audit_path = Path("/kaggle/working/rad_e10_audit.json")
    audit = {
        "status": "E2_PRESERVED",
        "mode": "pinned_v15_heads_inference_only",
        "experiment": "E10",
        "configuration": E10_CONFIG,
        "evidence_boundary": (
            "OOF values are 58-study diagnostics on official train labels, not Kaggle "
            "scores. E10 widens the weight ladder that E9 truncated at 0.25 and votes only "
            "where two independent OOF runs agree at the deployed weight."
        ),
        "encoder": "RadImageNet ResNet-50 official PyTorch release",
        "encoder_license": "CC-BY-NC-SA-4.0",
        "heads_sha256": PINNED_HEADS_SHA256,
        "contract_sha256": PINNED_E10_CONTRACT_SHA256,
        "parent": "E2 captured 20-member DINOv2 rank ensemble",
        "pixel_rules": dict(RULES),
    }
    if not primary.is_file():
        raise FileNotFoundError("E2 parent submission is absent")
    shutil.copy2(primary, preserved)

    try:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        if device.type != "cuda":
            raise RuntimeError("E10 RadImageNet inference requires CUDA")
        elapsed = max(0.0, time.time() - float(globals().get("T0", time.time())))
        available = 8.72 * 3600 - elapsed
        audit["elapsed_before_e10_seconds"] = elapsed
        audit["available_at_start_seconds"] = available
        if available < 45 * 60:
            raise TimeoutError(f"only {available / 60:.1f} minutes remain")

        pinned = _v52_load_e10()
        audit["weight_gate"] = {
            "configuration": pinned["configuration"],
            "alpha_map": pinned["alpha_map"],
            "preserved_targets": pinned["preserved_targets"],
            "diagnostic_macro": pinned["diagnostic_macro"],
            "recomputed_macro": pinned["recomputed_macro"],
            "all_dual_positive": pinned["all_dual_positive"],
            "base_gold_macro_auc": pinned["contract"]["base_gold_macro_auc"],
            "rationale": pinned["rationale"],
            "remote_ladder_recomputed_in_kernel": True,
        }

        test = pd.read_csv(ROOT / "test.csv", dtype={"StudyInstanceUID": str})
        rad_test, token_count, head_count = _v52_rad_test_predictions(
            pinned, test, device, "test-e10"
        )

        baseline = pd.read_csv(preserved, dtype={"StudyInstanceUID": str})
        _v52_validate_submission(baseline, test.StudyInstanceUID)
        rad_frame = pd.DataFrame(rad_test, columns=TARGETS)
        rad_frame.insert(0, "StudyInstanceUID", test.StudyInstanceUID)
        _v52_validate_submission(rad_frame, test.StudyInstanceUID)
        rad_frame.to_csv(output / "submission_rad_only.csv", index=False)
        baseline_rank = _v52_rank_columns(baseline[TARGETS].to_numpy())
        rad_rank = _v52_rank_columns(rad_test)

        # Materialise every audited rung so the ladder is inspectable from one run; only the
        # configured map is promoted to the visible submission.
        written = {}
        for name, configuration in sorted(pinned["contract"]["configurations"].items()):
            frame = baseline.copy()
            for target, alpha in configuration["alpha_map"].items():
                alpha = float(alpha)
                if alpha > 0:
                    index = TARGETS.index(target)
                    frame[target] = (
                        (1.0 - alpha) * baseline_rank[:, index] + alpha * rad_rank[:, index]
                    )
            for target, alpha in configuration["alpha_map"].items():
                if float(alpha) == 0.0 and not np.array_equal(
                    frame[target].to_numpy(), baseline[target].to_numpy()
                ):
                    raise RuntimeError(f"E10 failed to preserve {target} in {name}")
            _v52_validate_submission(frame, test.StudyInstanceUID)
            path = output / f"submission_e10_{name}.csv"
            frame.to_csv(path, index=False)
            written[name] = _v52_sha256(path)
        audit["ladder_sha256"] = written

        selected_path = output / f"submission_e10_{pinned['configuration']}.csv"
        selected = pd.read_csv(selected_path, dtype={"StudyInstanceUID": str})
        _v52_validate_submission(selected, test.StudyInstanceUID)
        for target in pinned["preserved_targets"]:
            if not np.array_equal(
                selected[target].to_numpy(), baseline[target].to_numpy()
            ):
                raise RuntimeError(f"E10 promoted file does not preserve {target}")
        audit.update({
            "test_studies": len(test),
            "test_available_slice_tokens": token_count,
            "test_head_count": head_count,
            "selected_path": str(selected_path),
            "selected_sha256": _v52_sha256(selected_path),
            "fallback_sha256": _v52_sha256(preserved),
        })
        shutil.copy2(selected_path, primary)
        if _v52_sha256(primary) != audit["selected_sha256"]:
            raise RuntimeError("primary E10 copy hash mismatch")
        audit["status"] = "CANDIDATE_SELECTED"
        voted = sorted(t for t, alpha in pinned["alpha_map"].items() if alpha > 0)
        log(
            f"E10 promoted {pinned['configuration']} over {len(voted)} dual-OOF targets; "
            f"{', '.join(pinned['preserved_targets'])} preserve E2"
        )
    except Exception as error:
        audit["status"] = "ERROR_E2_PRESERVED"
        audit["error"] = f"{type(error).__name__}: {error}"
        audit["traceback"] = traceback.format_exc()
        log(f"E10 preserves E2: {audit['error']}")
    finally:
        if audit.get("status") != "CANDIDATE_SELECTED" and preserved.is_file():
            shutil.copy2(preserved, primary)
        audit["primary_sha256"] = _v52_sha256(primary) if primary.is_file() else None
        audit_path.write_text(json.dumps(audit, indent=2, sort_keys=True) + "\n")


def _v52_e11_availability(headers, plane_map):
    """Count studies offering each (plane, fat-suppression) pair before any slot is picked.

    The parent arm reads only fat-suppressed series and never had to ask how many studies
    carry a non-suppressed one. E11 depends on that answer, so it is measured and logged
    rather than assumed: a slot nobody can fill is a masked column, and a run that produced
    one silently would look like a weak arm instead of an absent input.
    """
    frame = headers[["StudyInstanceUID", "SeriesInstanceUID", "fatsat"]].copy()
    frame["plane"] = frame.SeriesInstanceUID.map(plane_map)
    total = frame.StudyInstanceUID.nunique()
    table = {}
    for plane in ("Sagittal", "Coronal", "Axial"):
        for fatsat in (True, False):
            selected = frame[(frame.plane == plane) & (frame.fatsat == bool(fatsat))]
            studies = selected.StudyInstanceUID.nunique()
            key = f"{plane}_{'FS' if fatsat else 'NOFS'}"
            table[key] = {
                "studies": int(studies),
                "fraction": float(studies / total) if total else 0.0,
                "series": int(len(selected)),
            }
            log(f"E11 availability {key}: {studies}/{total} studies, {len(selected)} series")
    return table


def main_v52_e11():
    """Train a third arm on a deliberately different pixel recipe. Never ships a candidate."""
    import shutil
    from sklearn.model_selection import GroupKFold

    output = Path("/kaggle/working/rsna_rad_e11")
    output.mkdir(parents=True, exist_ok=True)
    primary = Path("/kaggle/working/submission.csv")
    preserved = Path("/kaggle/working/submission_e2_preserved.csv")
    audit_path = Path("/kaggle/working/rad_e11_audit.json")
    audit = {
        "status": "E2_PRESERVED",
        "mode": "e11-diverse-recipe-training-only",
        "evidence_boundary": (
            "Every value here is a local out-of-fold diagnostic on 58 official image "
            "labels. It is not a Kaggle score, and this mode never replaces the parent "
            "submission under any outcome."
        ),
        "recipe": {
            "slots": [list(slot) for slot in E11_SLOTS],
            "crop_mm": E11_CROP_MM,
            "cache_slices": E11_CACHE_SLICES,
            "img": E11_IMG,
            "differs_from_parent_arm": (
                "parent reads 3 fat-suppressed slots at full frame; this reads 3 "
                "non-suppressed slots plus 1 suppressed anchor at a 130 mm physical crop"
            ),
        },
        "encoder": "RadImageNet ResNet-50 official PyTorch release",
        "encoder_license": "CC-BY-NC-SA-4.0 (Kaggle-hosted weight metadata)",
        "encoder_sha256": "08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734",
    }
    if not primary.is_file():
        raise FileNotFoundError("E2 parent submission is absent")
    shutil.copy2(primary, preserved)

    try:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        if device.type != "cuda":
            raise RuntimeError("E11 training requires CUDA")
        elapsed = max(0.0, time.time() - float(globals().get("T0", time.time())))
        available = TIME_BUDGET - elapsed
        audit["elapsed_before_e11_seconds"] = elapsed
        audit["available_at_start_seconds"] = available
        if available < 2.5 * 3600:
            raise TimeoutError(f"only {available / 60:.1f} minutes remain")

        # Point the shared pixel path at the E11 recipe. These are the same process globals
        # the parent notebook's readers consult, so the override has to happen before any
        # slot is picked or any pixel is decoded, and nothing after this point may assume
        # the parent arm's values.
        globals().update(
            SLOTS=list(E11_SLOTS),
            N_SLOT=len(E11_SLOTS),
            CACHE_SLICES=int(E11_CACHE_SLICES),
            IMG=int(E11_IMG),
            CACHE_IMG=int(E11_IMG),
            CROP_MM=float(E11_CROP_MM),
        )
        log(
            f"E11 recipe: {[s[0] for s in E11_SLOTS]} at {E11_CROP_MM:.0f} mm, "
            f"{E11_IMG} px, {E11_CACHE_SLICES} slices/slot"
        )

        train = pd.read_csv(ROOT / "train.csv", dtype={"StudyInstanceUID": str})
        train_series = pd.read_csv(
            ROOT / "train_series.csv",
            dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
        )
        if len(train) != 4407:
            raise RuntimeError(f"unexpected train study count {len(train)}")
        plane = dict(zip(train_series.SeriesInstanceUID, train_series.Anatomical_Plane))
        headers = annotate(walk("train_series"))
        audit["availability"] = _v52_e11_availability(headers, plane)

        studies, pixels, slot_mask = build_cache(
            pick_slots(headers, plane), plane, lat_of(headers, "train-e11 "), "train-e11"
        )
        by_uid = {str(uid): i for i, uid in enumerate(studies)}
        missing = [uid for uid in train.StudyInstanceUID if uid not in by_uid]
        if missing:
            raise RuntimeError(f"{len(missing)} train studies absent from cache")
        order = np.array([by_uid[uid] for uid in train.StudyInstanceUID], dtype=np.int64)
        pixels, slot_mask = pixels[order], slot_mask[order]
        fill = float(slot_mask.mean())
        per_slot = {
            name: float(slot_mask[:, k].mean())
            for k, (name, _, _, _) in enumerate(E11_SLOTS)
        }
        audit["slot_fill"] = per_slot
        audit["overall_fill"] = fill
        for name, value in per_slot.items():
            log(f"E11 slot fill {name}: {value:.1%}")
        if fill < E11_MIN_FILL:
            raise RuntimeError(f"E11 slot fill {fill:.1%} below the {E11_MIN_FILL:.0%} floor")

        features, token_mask = encode_radimagenet(pixels, slot_mask, device)
        del pixels, slot_mask, headers
        gc.collect()

        y, weights, gold = make_targets(train)
        if int(gold.sum()) != 58:
            raise RuntimeError(f"expected 58 fully gold studies, observed {int(gold.sum())}")
        groups = report_groups(train)
        if len(np.unique(groups)) < 4000:
            raise RuntimeError("unexpected report-group collapse")

        splits = list(GroupKFold(5).split(features, groups=groups))
        fold_id = np.full(len(train), -1, dtype=np.int8)
        folds = []
        oof = np.zeros_like(y, dtype=np.float32)
        for fold, (tr, va) in enumerate(splits):
            if set(groups[tr]).intersection(groups[va]):
                raise RuntimeError(f"report leakage in fold {fold}")
            fold_id[va] = fold
            state, score = train_fold(
                features, token_mask, y, weights, tr, va, fold, device
            )
            if state is None:
                raise RuntimeError(f"fold {fold} produced no checkpoint")
            head = FoundationQueryHead().to(device)
            head.load_state_dict(state, strict=True)
            oof[va] = predict_head(head, features, token_mask, va, device)
            folds.append({"fold": fold, "weak_auc": float(score), "state_dict": state})
            del head
            torch.cuda.empty_cache()
        if (fold_id < 0).any() or not np.isfinite(oof).all():
            raise RuntimeError("incomplete E11 OOF")

        weak_auc = macro_auc(y, oof)
        gold_auc = macro_auc(y[gold], oof[gold])
        audit["weak_oof_auc"] = float(weak_auc)
        audit["gold_oof_auc"] = float(gold_auc)
        log(f"E11 OOF weak macro AUC {weak_auc:.5f}")
        log(f"E11 OOF gold macro AUC {gold_auc:.5f} on 58 studies")

        # The question E11 exists to answer is not whether this arm is strong on its own but
        # whether it says something the portfolio does not already know. Both halves are
        # measured against the same 58 rows and the same rank basis the deployed blend uses.
        base_npz = find_input_file("oof.npz")
        with np.load(base_npz, allow_pickle=False) as bundle:
            if bundle["targets"].astype(str).tolist() != TARGETS:
                raise RuntimeError("E11 E2 OOF target order drift")
            if not np.array_equal(
                bundle["ids"].astype(str), train.StudyInstanceUID.astype(str).to_numpy()
            ):
                raise RuntimeError("E11 E2 OOF study order drift")
            base_prediction = bundle["pred"].astype(np.float64)
        base = _v52_rank_columns(base_prediction[gold])
        new = _v52_rank_columns(oof[gold].astype(np.float64))
        gold_y = train.loc[gold, TARGETS].to_numpy(np.float64)
        reference = _v52_target_auc(gold_y, base)
        audit["e2_base_gold_macro"] = float(np.mean([reference[t] for t in TARGETS]))
        ladder = {}
        for alpha in (0.20, 0.35, 0.50):
            scores = _v52_target_auc(gold_y, (1.0 - alpha) * base + alpha * new)
            ladder[f"{alpha:.2f}"] = {
                "macro": float(np.mean([scores[t] for t in TARGETS])),
                "per_target_delta": {t: float(scores[t] - reference[t]) for t in TARGETS},
            }
            log(f"E11 blend alpha={alpha:.2f} gold macro {ladder[f'{alpha:.2f}']['macro']:.5f}")
        audit["blend_vs_e2"] = ladder

        try:
            parent_oof = pd.read_csv(
                find_input_file("v52_oof.csv"), dtype={"StudyInstanceUID": str}
            )
            aligned = train[["StudyInstanceUID"]].merge(
                parent_oof, on="StudyInstanceUID", how="left", validate="one_to_one"
            )
            parent = _v52_rank_columns(aligned[TARGETS].to_numpy(np.float64)[gold])
            audit["correlation_with_parent_arm"] = {
                t: float(np.corrcoef(parent[:, i], new[:, i])[0, 1])
                for i, t in enumerate(TARGETS)
            }
            log(
                "E11 mean rank correlation with the parent arm: "
                f"{np.mean(list(audit['correlation_with_parent_arm'].values())):.3f}"
            )
        except FileNotFoundError:
            audit["correlation_with_parent_arm"] = None

        torch.save(
            {
                "version": "e11-radimagenet-resnet50-diverse-1",
                "targets": TARGETS,
                "encoder_sha256": audit["encoder_sha256"],
                "slots": [list(slot) for slot in E11_SLOTS],
                "crop_mm": E11_CROP_MM,
                "img": E11_IMG,
                "slices_per_plane": E11_CACHE_SLICES,
                "feature": "global_average_pool",
                "folds": folds,
                "weak_oof_auc": float(weak_auc),
                "gold_oof_auc": float(gold_auc),
            },
            output / "v52_e11_heads.pt",
        )
        oof_frame = pd.DataFrame(oof, columns=TARGETS)
        oof_frame.insert(0, "StudyInstanceUID", train.StudyInstanceUID)
        oof_frame["fold"] = fold_id
        oof_frame["is_gold"] = gold.astype(np.uint8)
        oof_frame.to_csv(output / "v52_e11_oof.csv", index=False)
        audit["status"] = "E11_TRAINED_E2_PRESERVED"
        audit["heads_sha256"] = _v52_sha256(output / "v52_e11_heads.pt")
        audit["oof_sha256"] = _v52_sha256(output / "v52_e11_oof.csv")
    except Exception as error:
        audit["status"] = "ERROR_E2_PRESERVED"
        audit["error"] = f"{type(error).__name__}: {error}"
        audit["traceback"] = traceback.format_exc()
        log(f"E11 preserves E2: {audit['error']}")
    finally:
        if preserved.is_file():
            shutil.copy2(preserved, primary)
        audit["primary_sha256"] = _v52_sha256(primary) if primary.is_file() else None
        audit_path.write_text(json.dumps(audit, indent=2, sort_keys=True) + "\n")


if ARM_MODE == "s0":
    pass  # track S: builder cell below
elif ARM_MODE == "e11":
    main_v52_e11()
else:
    try:
        find_input_file("v52_radimagenet_heads.pt")
    except FileNotFoundError:
        main_v52()
    else:
        try:
            find_input_file("e10_contract.json")
        except FileNotFoundError:
            main_v52_pinned_e9b()
        else:
            main_v52_e10()



In [ ]:
# ===== Track S / Batch 2: build a cache VARIANT on CPU (no GPU quota) =====
# knee-autoresearch, 2026-09-19. Appended after E13-train cells 0-17 + 32 (helpers only; the DINOv2/DINOv3 pipeline
# cells 23-31 are left out). Same studies, slots (SAG_FS/COR_FS/AX_FS/SAG_NOFS), 8 slices, 224 px, SLICE_BAND
# (0.12, 0.88) as S0; one recipe knob changes per variant. Labels / weights / folds are unchanged, so
# s1_train_arrays.npz and ResNet50.pt are copied from the S0 output (attached as input) -> the output dir is a
# complete --data dir for s1_train_vast.py.
import shutil

VARIANT = {'name': 'sl16', 'crop_mm': 130.0, 'rules': 'legacy', 'slices': 16}   # filled per notebook
OUT = Path('/kaggle/working/track_s_' + VARIANT['name'])


def main_s0b():
    OUT.mkdir(parents=True, exist_ok=True)
    rules = dict(RULES_NATIVE if VARIANT['rules'] == 'native' else RULES_LEGACY)
    globals().update(SLOTS=list(E11_SLOTS), N_SLOT=len(E11_SLOTS), CACHE_SLICES=int(VARIANT['slices']), IMG=224, CACHE_IMG=224,
                     CROP_MM=float(VARIANT['crop_mm']), SLICE_BAND=(0.12, 0.88), RULES=rules)
    log(f'[S0b] variant {VARIANT} | rules {RULES} | band {SLICE_BAND}')
    s0 = None
    for root, dirs, files in os.walk('/kaggle/input'):
        dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series')]
        if 's1_train_arrays.npz' in files and 'ResNet50.pt' in files:
            s0 = Path(root)
    assert s0 is not None, 'attach the rsna-knee-s0-cache output'
    with np.load(s0 / 's1_train_arrays.npz', allow_pickle=False) as a:
        ids = a['ids'].astype(str)
    train = pd.read_csv(ROOT / 'train.csv', dtype={'StudyInstanceUID': str})
    if not np.array_equal(train.StudyInstanceUID.astype(str).to_numpy(), ids):
        raise RuntimeError('train.csv order differs from the S0 arrays')
    train_series = pd.read_csv(ROOT / 'train_series.csv', dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
    plane = dict(zip(train_series.SeriesInstanceUID, train_series.Anatomical_Plane))
    headers = annotate(walk('train_series'))
    tag = 'train-' + VARIANT['name']
    studies, pixels, slot_mask = build_cache(pick_slots(headers, plane), plane, lat_of(headers, tag + ' '), tag)
    by_uid = {str(u): i for i, u in enumerate(studies)}
    missing = [u for u in ids if u not in by_uid]
    if missing:
        raise RuntimeError(f'{len(missing)} train studies absent from the cache')
    order = np.array([by_uid[u] for u in ids], dtype=np.int64)
    # Reorder to train.csv order straight into the output file (a 16-slice cache is 14 GB: no second copy in RAM).
    out = np.lib.format.open_memmap(OUT / 's_cache_e13_224.npy', mode='w+', dtype=np.uint8,
                                    shape=(len(ids),) + tuple(pixels.shape[1:]))
    for c in range(0, len(order), 256):
        out[c:c + 256] = pixels[order[c:c + 256]]
    out.flush()
    shape = tuple(out.shape)
    del out, pixels
    gc.collect()
    slot_mask = np.asarray(slot_mask)[order].astype(np.float32)
    fill = {s[0]: float(slot_mask[:, j].mean()) for j, s in enumerate(E11_SLOTS)}
    log(f'[S0b] cache {shape}; slot fill {fill}')
    np.savez(OUT / 's_cache_e13_224_meta.npz', ids=ids, slot_mask=(slot_mask > 0).astype(np.uint8),
             slots=np.array([s[0] for s in E11_SLOTS]), crop_mm=np.float32(CROP_MM), img=np.int32(224),
             slices=np.int32(VARIANT['slices']), rules=np.array(json.dumps(dict(RULES))))
    for name in ('s1_train_arrays.npz', 'ResNet50.pt', 's1_frozen_oof.csv'):
        shutil.copy2(s0 / name, OUT / name)
    audit = {'variant': VARIANT, 'rules': dict(RULES), 'slice_band': list(SLICE_BAND), 'shape': list(shape),
             'slot_fill': fill, 'files': {p.name: p.stat().st_size for p in sorted(OUT.iterdir())}}
    (OUT / 's0b_audit.json').write_text(json.dumps(audit, indent=1))
    print(json.dumps(audit, indent=1), flush=True)


main_s0b()
